In [ ]:
%pip install pylatexenc
%pip install -U langchain-google-vertexai
%pip install -U langchain-community
%pip install -U langchain-huggingface
%pip install faiss-gpu
#%pip install faiss-cpu
%pip install regex
%pip install chardet
%pip install pytictoc
%pip install --upgrade google-genai

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.6 MB/s eta 0:00:00
  Attempting uninstall: langchain-community
    Found existing installation: langchain-community 0.3.22
    Uninstalling langchain-community-0.3.22:
      Successfully uninstalled langchain-community-0.3.22
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 96.2 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [1]:
import tarfile
import zipfile
import io
import os
import time
import math
import pickle
import itertools as itr
import functools as ft
import regex as re
import chardet
from tqdm.auto import tqdm
from pytictoc import TicToc
import json

import pandas as pd
import numpy as np

import gcsfs
fs = gcsfs.GCSFileSystem()

from google.cloud import storage
from google.cloud.exceptions import ClientError

PROJECT_ID = "arxiv-development"
PRD_PROJECT = 'arxiv-production'
PRD_BUCKET_LOC = 'arxiv-production-data' 

from pylatexenc.latexwalker import LatexWalker, LatexEnvironmentNode, LatexGroupNode, LatexMacroNode, LatexCharsNode
from pylatexenc.latex2text import LatexNodes2Text


In [2]:
os.chdir("/home/jupyter/metadata-vertexai/")  # this needs to be the folder where notebook lives
import importlib
import phase_one_json as phase_one

In [3]:
from IPython.core.interactiveshell import InteractiveShell
# pretty print all cell's output and not just the last one
InteractiveShell.ast_node_interactivity = "all"

In [4]:
def safe_divide(num, denom):
    return num / denom if denom != 0 else 0.0

Note that id lists were prepared previously from the DB, using:  

```
select concat(paper_id,"v",version) as arx_id from arXiv_metadata
where paper_id LIKE "23%";
and is_withdrawn != 1
and is_current = 1;
```

In [5]:
ror_gspath = 'gs://institutional-extract-scratch/reference/v1.63-2025-04-03-ror-data_schema_v2.json'
fs = gcsfs.GCSFileSystem()
with fs.open(ror_gspath, "r", encoding="utf-8") as f:
    ror_data = json.load(f)
ror_dict = {x['id'].rsplit('/')[-1]: x for x in ror_data}

In [6]:
#ror_dict['043mz5j54']

In [7]:
def get_children_for_ror(target_id):
    res_list = []
    rels = ror_dict[target_id].get('relationships',[])
    for rel in rels:
        rel_type = rel.get('type','')
        if rel_type != 'child':
            continue
        child_id = rel.get('id','').rsplit('/')[-1]
        if child_id:
            res_list.append(child_id)
    return res_list


## Set parameters

In [10]:
#results_file = "gs://institutional-extract-scratch/reference/arx_ids/2311_ids.csv"
results_file = "gs://institutional-extract-scratch/output/2311_db_all_2025-04-28.csv.zip"

In [11]:
test_ids_df = pd.read_csv(results_file)

test_ids_df.head()
ids_2311_all = test_ids_df["arx_id"].unique()

,arx_id,name,ror
0,2311.07251v1,"TU Dortmund University, Dortmund",01k97gp34
1,2311.08209v2,"University of Tokyo,",057zh3y96
2,2311.04723v3,"Nanjing University, Nanjing",01rxvg760
3,2311.04723v3,Hefei National Laboratory,02mp2av58
4,2311.02334v1,"Duke University, Durham",00py81415


In [12]:
test_ids_df.shape

(48360, 3)

In [13]:
vip_df = pd.read_csv("gs://institutional-extract-scratch/reference/dashboard_institutions2024_2025-05-01.csv", dtype=str)
vip_df = vip_df[vip_df['is_consortium']=='0']
vip_df.shape
vip_df.head()

(345, 21)

,sid,name,country,country_code,consortia_code,member_type,ror_id,sub_rors,is_consortium,label,...,is_active,Institution,salsaId,orgId,Comment,Usage Contact Email,First Name,Last Name,Title,Unnamed: 17
0,447,Aalto University,Finland,FI,FinELib,member,https://ror.org/020hwjq30,NaN,0,Aalto University,...,1,Aalto University,52318446,60103653,NaN,antti.m.rousi@aalto.fi,Antti,Rousi,"Specialist, Research Services",NaN
1,465,Abo Akademi University,Finland,FI,FinELib,member,https://ror.org/029pk6x14,NaN,0,Abo Akademi University,...,1,Abo Akademi University,NaN,60015375,No contact for school this is the consortium c...,timo.vilen@helsinki.fi,Timo,Vilén,Information Specialist,NaN
2,482,Ames Laboratory,United States,US,NaN,member,https://ror.org/041m9xr71,NaN,0,Ames Laboratory,...,1,Ames Laboratory,52320619,60008023,NaN,lgraves@iastate.edu,Laura,Graves,NaN,NaN
3,483,Argonne National Lab,United States,US,NaN,member,https://ror.org/05gvnxz63,NaN,0,Argonne National Lab,...,1,Argonne National Lab,1714199,60028609,NaN,mstraka@anl.gov,Mary,Straka,NaN,NaN
4,16,Australian National University,Australia,AU,CAUL,member,https://ror.org/019wvm592,NaN,0,Australian National University,...,1,Australian National University,2090464,60008950,NaN,electronic.coordinator@anu.edu.au,NaN,NaN,NaN,NaN


In [14]:
vip_df['ror_id'].value_counts()
vip_df['orgId'].value_counts()

ror_id
https://ror.org/04tsk2644    2
https://ror.org/03nawhv43    1
https://ror.org/00d9ah105    1
https://ror.org/046rm7j60    1
https://ror.org/04gyf1771    1
                            ..
https://ror.org/041kmwe10    1
https://ror.org/03gnh5541    1
https://ror.org/03v8tnc06    1
https://ror.org/01hcx6992    1
https://ror.org/04ka0vh05    1
Name: count, Length: 338, dtype: int64

orgId
60005322    2
60029526    1
60027550    1
60007278    1
60014439    1
           ..
60117390    1
60000762    1
60014652    1
60030788    1
60032005    1
Name: count, Length: 339, dtype: int64

In [15]:
vip_df[vip_df['orgId'] == "60015150"]
vip_df[vip_df['name'].str.contains("Paris")]

,sid,name,country,country_code,consortia_code,member_type,ror_id,sub_rors,is_consortium,label,...,is_active,Institution,salsaId,orgId,Comment,Usage Contact Email,First Name,Last Name,Title,Unnamed: 17
122,570,"Imperial College of Science, Technology, and M...",United Kingdom,GB,NaN,champion,https://ror.org/041kmwe10,NaN,0,"Imperial College of Science, Technology, and M...",...,1,"Imperial College of Science, Technology, and M...",NaN,60015150,NaN,"libsubs@imperial.ac.uk,robyn.price@imperial.ac.uk",NaN,NaN,NaN,NaN


,sid,name,country,country_code,consortia_code,member_type,ror_id,sub_rors,is_consortium,label,...,is_active,Institution,salsaId,orgId,Comment,Usage Contact Email,First Name,Last Name,Title,Unnamed: 17
7,235,Bibliothèque de l'Observatoire de Paris (OBSPM),France,FR,CCSD,member,https://ror.org/029nkcm90,NaN,0,Bibliothèque de l'Observatoire de Paris (OBSPM),...,1,Bibliothèque de l'Observatoire de Paris (OBSPM),NaN,60003674,NaN,NaN,NaN,NaN,NaN,NaN
181,596,Paris Nanterre University,France,FR,COUP,member,https://ror.org/013bkhk48,NaN,0,Paris Nanterre University,...,1,Paris Nanterre University,NaN,60017338,NaN,mleguenn@parisnanterre.fr,NaN,NaN,NaN,NaN
333,653,Université Paris-Saclay,France,FR,CCSD,member,https://ror.org/03xjwb503,028rypz17,0,Université Paris-Saclay,...,1,Université Paris-Saclay,NaN,60106017,NaN,NaN,NaN,NaN,NaN,NaN
334,654,Université Paris-Sud,France,FR,CCSD,member,https://ror.org/028rypz17,03xjwb503),0,Université Paris-Sud,...,1,Université Paris-Sud,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
vip_df.head()['ror_id'].apply(lambda x: x.rsplit('/')[-1])

0    020hwjq30
1    029pk6x14
2    041m9xr71
3    05gvnxz63
4    019wvm592
Name: ror_id, dtype: object

In [17]:
ror_df = pd.DataFrame({
    'name':vip_df['name'],
    'ror':vip_df['ror_id'].apply(lambda x: pd.NA if pd.isna(x) else x.rsplit('/')[-1])
})
ror_df = ror_df.dropna()
ror_df.shape
ror_df.head()

(339, 2)

,name,ror
0,Aalto University,020hwjq30
1,Abo Akademi University,029pk6x14
2,Ames Laboratory,041m9xr71
3,Argonne National Lab,05gvnxz63
4,Australian National University,019wvm592


### Eval results

In [18]:
ror_map_df = pd.read_csv("gs://institutional-extract-scratch/reference/matched_results_ror_api.csv", dtype=str)
ror_map_df = ror_map_df.set_index('Primary Org Id')
ror_map_df.head()
ror_map_df[ror_map_df['Primary Org Name'].str.contains("Paris")].head(30)

,Primary Org Name,Country Name,ROR ID
Primary Org Id,,,
60000009,Villanova University,United States,https://ror.org/02g7kd627
60000011,Saitama Institute of Technology,Japan,https://ror.org/01pkeax38
60000015,Lusófona University,Portugal,https://ror.org/05xxfer42
60000021,Atatürk Üniversitesi,Turkey,https://ror.org/03je5c526
60000027,KLA Corporation,United States,https://ror.org/04zdyxh40


,Primary Org Name,Country Name,ROR ID
Primary Org Id,,,
60000195,Groupe Hospitalier Paris Saint-Joseph,France,https://ror.org/046bx1082
60000377,Universite de la Sorbonne Nouvelle Paris III,France,https://ror.org/01d8kr740
60001772,"Institut Pasteur, Paris",France,https://ror.org/0495fxg12
60001970,Institut de Physique du Globe de Paris,France,https://ror.org/004gzqz66
60002272,CY Cergy Paris Université,France,https://ror.org/043htjv09
60003674,L'Observatoire de Paris,France,https://ror.org/029nkcm90
60004833,Institut de Mathématiques de Jussieu-Paris Riv...,France,https://ror.org/03fk87k11
60004885,Universite Paris 8 Vincennes-St Denis,France,https://ror.org/04wez5e68
60004903,ParisTech,France,https://ror.org/05c2qg481


In [19]:
vip_df = vip_df.set_index('orgId')

In [20]:
vip_df.loc['60010365']

sid                                               411
name                   University of British Columbia
country                                        Canada
country_code                                       CA
consortia_code                                    NaN
member_type                                    member
ror_id                      https://ror.org/03rmrcq20
sub_rors                                          NaN
is_consortium                                       0
label                  University of British Columbia
comment                                           NaN
is_active                                           1
Institution            University of British Columbia
salsaId                                       1714346
Comment                                           NaN
Usage Contact Email                kat.mcgrath@ubc.ca
First Name                                        Kat
Last Name                                     McGrath
Title                       

In [21]:
ror_map_df = pd.merge(
    ror_map_df, 
    vip_df[['ror_id']],
    how='left', left_index=True, right_index=True,
)
ror_map_df['ror_id'] = ror_map_df['ror_id'].fillna(ror_map_df['ROR ID'])
ror_map_df.drop(columns=['ROR ID'], inplace=True)
ror_map_df['ror'] = ror_map_df['ror_id'].apply(lambda x: pd.NA if pd.isna(x) else x.rsplit('/')[-1])
rors_in_map = set(ror_map_df['ror'].unique())
ror_map_df = ror_map_df.reset_index()
ror_map_df.shape
ror_map_df.head()

(13366, 5)

,Primary Org Id,Primary Org Name,Country Name,ror_id,ror
0,60000009,Villanova University,United States,https://ror.org/02g7kd627,02g7kd627
1,60000011,Saitama Institute of Technology,Japan,https://ror.org/01pkeax38,01pkeax38
2,60000015,Lusófona University,Portugal,https://ror.org/05xxfer42,05xxfer42
3,60000021,Atatürk Üniversitesi,Turkey,https://ror.org/03je5c526,03je5c526
4,60000027,KLA Corporation,United States,https://ror.org/04zdyxh40,04zdyxh40


In [22]:
ror_map_df[ror_map_df['Primary Org Name'].str.contains("Paris-Sud")]

,Primary Org Id,Primary Org Name,Country Name,ror_id,ror


In [23]:
ror_map_df.index

RangeIndex(start=0, stop=13366, step=1)

In [24]:
vip_df[vip_df['name'].str.contains("George Washington")]
ror_map_df[ror_map_df['Primary Org Name'].str.contains("George Washington")]

,sid,name,country,country_code,consortia_code,member_type,ror_id,sub_rors,is_consortium,label,comment,is_active,Institution,salsaId,Comment,Usage Contact Email,First Name,Last Name,Title,Unnamed: 17
orgId,,,,,,,,,,,,,,,,,,,,
60003088,610,The George Washington University,NaN,NaN,NaN,member,https://ror.org/00y4zzh67,NaN,0,The George Washington University,NaN,1,The George Washington University,NaN,NaN,jel@gwu.edu,NaN,NaN,NaN,NaN


,Primary Org Id,Primary Org Name,Country Name,ror_id,ror
613,60003088,The George Washington University,United States,https://ror.org/00y4zzh67,00y4zzh67


In [25]:
scopus_df = pd.read_csv("gs://institutional-extract-scratch/training/2311_scopus_17416.csv.zip", dtype=str)
scopus_all = set(scopus_df["ArXiv Id"].unique())
scopus_positive = scopus_df[scopus_df['Primary Org Id'] == 60027550]["ArXiv Id"].unique()

scopus_df = pd.merge(
    scopus_df, 
    ror_map_df[["Primary Org Id", "ror"]], 
    how='left',
    on='Primary Org Id'
)
scopus_all = set(scopus_df["paper_id"].unique())

In [26]:
scopus_df['paper_id'].nunique()
scopus_df.shape
scopus_df.head()

17416

(80422, 9)

,Primary Org Id,Primary Org Name,Primary Org City,Primary Org State,Primary Org Country,ArXiv Id,Affiliation Sequence Number,paper_id,ror
0,60006297,University of Pennsylvania,Philadelphia,PA,United States,2311.03477v1,1,2311.03477,00b30xv10
1,60024190,"Institute of Plasma Physics, Academy of Scienc...",Prague,NaN,Czech Republic,2311.04187v1,9,2311.04187,01h494015
2,60025641,Universität Freiburg,Freiburg im Breisgau,Baden-Wurttemberg,Germany,2311.04557v1,1,2311.04557,0245cg223
3,60114755,"Istituto Nazionale di Fisica Nucleare, Sezione...",Milan,NaN,Italy,2311.14088v1,27,2311.14088,04w4m6z96
4,60115855,Trento Institute for Fundamental Physics and A...,Povo,TN,Italy,2311.09750v1,2,2311.09750,00nhs3j29


In [27]:
afile_df = pd.read_csv( "paris-sud.csv", dtype=str, index_col=None, usecols=['arx_id','name','ror'])
afile_df.head()
afile_df.isnull().sum()

,arx_id,name,ror
0,2311.13936v1,NaN,NaN
1,2311.11443v2,University of Oldenburg,033n9gh91
2,2311.11443v2,Université Paris-Saclay,03xjwb503
3,2311.11443v2,INRIA,02kvxyf05
4,2311.11443v2,LMF,00gdtta79


arx_id     0
name      37
ror       97
dtype: int64

In [28]:
res_df = pd.read_csv(results_file)
append_files = [
    "paris-sud.csv",
    "king-college.csv",
]
for afile in append_files:
    afile_df = pd.read_csv(afile, dtype=str, index_col=None, usecols=['arx_id', 'name', 'ror'])
    res_df = pd.concat([res_df, afile_df], axis=0).reset_index(drop=True)
res_df.shape

res_df['paper_id'] = res_df['arx_id'].apply(lambda x: x.rsplit('v',1)[0])
res_df.shape
res_df.head()

(53453, 3)

(53453, 4)

,arx_id,name,ror,paper_id
0,2311.07251v1,"TU Dortmund University, Dortmund",01k97gp34,2311.07251
1,2311.08209v2,"University of Tokyo,",057zh3y96,2311.08209
2,2311.04723v3,"Nanjing University, Nanjing",01rxvg760,2311.04723
3,2311.04723v3,Hefei National Laboratory,02mp2av58,2311.04723
4,2311.02334v1,"Duke University, Durham",00py81415,2311.02334


In [29]:
skip_inst = {
    '01nsd7y51': 'CSIC - Geociencias Barcelona (GEO3BCN)',
    '05dsysc59': 'CSIC - Instituto de Carboquímica (ICB)',
    '04zdays56': 'CSIC - Instituto de Biologia Molecular y Celul...',
    '03hasqf61': 'CSIC - Institut de Ciència de Materials de Bar...',
    '02h7vfp25': 'CSIC - Instituto de Cerámica y Vidrio (ICV)',
    '04qayn356': 'CSIC - Instituto de Ciencias Marinas de Andalu...',
}

scopus_to_rerun = set()
vip_results = []
for ror in tqdm(ror_df['ror'].unique()):
    if ror in skip_inst:
        continue
    if not ror in rors_in_map:
        print(f"'{ror}': {ror_df.loc[ror_df['ror']==ror,'name'].iloc[0]}")
        continue
    inst_name = ror_map_df[ror_map_df['ror']==ror]['Primary Org Name'].iloc[0]
    sub_ids = get_children_for_ror(ror)
    sub_ids.append(ror)
    scopus_ids = set(scopus_df[scopus_df['ror'].isin(sub_ids)]['paper_id'].unique())
    scopus_arx_ids = set(scopus_df[scopus_df['ror'].isin(sub_ids)]['ArXiv Id'].unique())
    scopus_to_rerun = scopus_to_rerun.union(scopus_arx_ids)
    res_ids = set(res_df[res_df['ror'].isin(sub_ids)]['arx_id'].apply(lambda x: x.rsplit('v')[0]).unique())
    res_ids = res_ids.intersection(scopus_all) # only consider cases in the scopus data
    res = {
        "name": inst_name,
        "ror": ror,
        "TP": len(res_ids.intersection(scopus_ids)), 
        "FP": len(res_ids - scopus_ids), 
        "FN": len(scopus_ids - res_ids), 
        "TN": len((scopus_all - scopus_ids) - res_ids)
    }
    res['precision'] = safe_divide(res['TP'], res['TP'] + res['FP'])
    res['recall'] = safe_divide(res['TP'], res['TP'] + res['FN'])
    if (res['precision'] + res['recall']) > 0:
        res['F1'] = safe_divide(2 * res['precision'] * res['recall'], res['precision'] + res['recall'])
    else:
        res['F1'] = 0.0
    vip_results.append(res)


  0%|          | 0/338 [00:00<?, ?it/s]

'03srn9y98': CSIC - Instituto de Química Avanzada de Cataluña (IQAC)
'006gw6z14': CSIC- Estación Biológica de Doñana EBD
'04nrv3s86': CSIC-UMA - Instituto de Hortofruticultura Subtropical y Mediterranea La Mayora (IHSM)
'000nhpy59': CSIC-UMH - Instituto de Neurociencias (IN)
'04zp24820': Chennai Mathematical Institute
'021f7p178': Lib4RI
'052rrw050': National Astronomical Observatory of Japan
'049bh0z35': National Library of Sweden
'03kgj4539': TRIUMF
'00bwtjf83': Tampere University of Applied Sciences
'028rypz17': Université Paris-Sud


In [30]:
vip_res_df = pd.DataFrame.from_records(vip_results)
vip_res_df.head()

,name,ror,TP,FP,FN,TN,precision,recall,F1
0,Aalto University,020hwjq30,56,2,25,17333,0.965517,0.691358,0.805755
1,Åbo Akademi University,029pk6x14,2,0,0,17414,1.000000,1.000000,1.000000
2,Ames Laboratory,041m9xr71,4,0,2,17410,1.000000,0.666667,0.800000
3,Argonne National Laboratory,05gvnxz63,49,3,8,17356,0.942308,0.859649,0.899083
4,The Australian National University,019wvm592,70,4,8,17334,0.945946,0.897436,0.921053


In [31]:
len(scopus_to_rerun)


9736

In [32]:
#vip_res_df.to_csv("vip_results_2025-04-28.csv") 

In [33]:
total_gt_cases = vip_res_df[['TP', 'FP', 'FN', 'TN']].iloc[0].sum()

In [34]:
cutoff = 10
tn_min = total_gt_cases - cutoff
vip_res_df.query("TN <= @tn_min").sort_values('F1', ascending=True).head(20)
vip_res_df[vip_res_df['ror'].str.contains('03xjwb503')].sort_values('F1', ascending=True).head(10)

,name,ror,TP,FP,FN,TN,precision,recall,F1
181,Technische Universität Kaiserslautern,04zrf7b53,3,0,22,17391,1.000000,0.120000,0.214286
143,Foundation for Fundamental Research on Matter,00f9tz983,10,1,35,17370,0.909091,0.222222,0.357143
59,CSIC-UV - Instituto de Física Corpuscular,017xch102,7,0,25,17384,1.000000,0.218750,0.358974
272,"University of the Witwatersrand, Johannesburg",03rp50x72,9,2,22,17383,0.818182,0.290323,0.428571
69,Commissariat a l'Energie Atomique et aux Energ...,00jjx8s55,75,5,140,17196,0.937500,0.348837,0.508475
299,Université Toulouse III - Paul Sabatier,02v6kpv12,33,0,60,17323,1.000000,0.354839,0.523810
44,CSIC-UAM - Instituto de Física Teórica (IFT),022r8mj40,13,3,20,17380,0.812500,0.393939,0.530612
103,Institute of Physics of the Czech Academy of S...,02yhj4v17,13,0,22,17381,1.000000,0.371429,0.541667
309,Villanova University,02g7kd627,6,1,8,17401,0.857143,0.428571,0.571429
95,"National Science Library, Chinese Academy of S...",03v8tnc06,24,19,14,17359,0.558140,0.631579,0.592593


,name,ror,TP,FP,FN,TN,precision,recall,F1
298,Université Paris-Saclay,03xjwb503,197,4,117,17098,0.9801,0.627389,0.765049


In [35]:
scopus_df.head()

,Primary Org Id,Primary Org Name,Primary Org City,Primary Org State,Primary Org Country,ArXiv Id,Affiliation Sequence Number,paper_id,ror
0,60006297,University of Pennsylvania,Philadelphia,PA,United States,2311.03477v1,1,2311.03477,00b30xv10
1,60024190,"Institute of Plasma Physics, Academy of Scienc...",Prague,NaN,Czech Republic,2311.04187v1,9,2311.04187,01h494015
2,60025641,Universität Freiburg,Freiburg im Breisgau,Baden-Wurttemberg,Germany,2311.04557v1,1,2311.04557,0245cg223
3,60114755,"Istituto Nazionale di Fisica Nucleare, Sezione...",Milan,NaN,Italy,2311.14088v1,27,2311.14088,04w4m6z96
4,60115855,Trento Institute for Fundamental Physics and A...,Povo,TN,Italy,2311.09750v1,2,2311.09750,00nhs3j29


In [38]:
name_txt = "Université Toulouse"
scopus_df[scopus_df['Primary Org Name'].str.contains(name_txt)].head()
ror_map_df[ror_map_df['Primary Org Name'].str.contains(name_txt)]
vip_df[vip_df['name'].str.contains(name_txt)]

,Primary Org Id,Primary Org Name,Primary Org City,Primary Org State,Primary Org Country,ArXiv Id,Affiliation Sequence Number,paper_id,ror
31,60005899,Université Toulouse - Jean Jaurès,Toulouse,Occitanie,France,2311.08863v1,3,2311.08863,04ezk3x31
50,60005899,Université Toulouse - Jean Jaurès,Toulouse,Occitanie,France,2311.05631v1,1,2311.05631,04ezk3x31
104,60020551,Université Toulouse 1 Capitole,Toulouse,Occitanie,France,2311.02507v2,1,2311.02507,0443n9e75
2285,60027245,Université Toulouse III - Paul Sabatier,Toulouse,Occitanie,France,2311.07011v2,9,2311.07011,02v6kpv12
2294,60027245,Université Toulouse III - Paul Sabatier,Toulouse,Occitanie,France,2311.05455v1,1,2311.05455,02v6kpv12


,Primary Org Id,Primary Org Name,Country Name,ror_id,ror
1137,60005899,Université Toulouse - Jean Jaurès,France,https://ror.org/04ezk3x31,04ezk3x31
3890,60020551,Université Toulouse 1 Capitole,France,https://ror.org/0443n9e75,0443n9e75
5162,60027245,Université Toulouse III - Paul Sabatier,France,https://ror.org/02v6kpv12,02v6kpv12


,sid,name,country,country_code,consortia_code,member_type,ror_id,sub_rors,is_consortium,label,comment,is_active,Institution,salsaId,Comment,Usage Contact Email,First Name,Last Name,Title,Unnamed: 17
orgId,,,,,,,,,,,,,,,,,,,,
60027245,655,Université Toulouse III -Paul Sabatier,France,FR,COUP,member,https://ror.org/02v6kpv12,NaN,0,Université Toulouse III -Paul Sabatier,NaN,1,Université Toulouse III -Paul Sabatier,NaN,NaN,"pierre.naegelen@univ-tlse3.fr,deborah.florent@...",Pierre,Naegelene,NaN,NaN


In [39]:
ror = "02v6kpv12"
sub_ids = get_children_for_ror(ror)
sub_ids.append(ror)
print(sub_ids)
scopus_ids = set(scopus_df[scopus_df['ror'].isin(sub_ids)]['paper_id'].unique())
scopus_arx_ids = set(scopus_df[scopus_df['ror'].isin(sub_ids)]['ArXiv Id'].unique())
len(scopus_ids)
res_df[res_df['ror']==ror].shape
res_df[res_df['ror']==ror].head()

['04fhrs205', '003412r28', '02hbzmb19', '01225hq90', '034nb0f30', '045ktmd28', '05k0qmv73', '040smqw14', '025nmxp11', '016zvc994', '03xhggy77', '02ywmqv15', '02v2svk17', '017d9yp59', '003jnac13', '04rrj3a80', '047z5as19', '0171mae58', '02gwt2810', '05tcnbj64', '02chvqy57', '027rbaq21', '02xh23b55', '03vcm6439', '02w5mvk98', '03p7xrr08', '0111s2360', '05hm2ja81', '05d6wfd23', '014vp6c30', '0111a5077', '00j50jx72', '01bxfc994', '033z83z59', '03wa9cd25', '03tvs6n06', '02k7ask46', '00qhm9960', '00n90tt57', '03j65n394', '056g7f250', '02wq1s711', '01r19bq53', '045p8nc06', '03v2c3v44', '02v6kpv12']


93

(18, 4)

,arx_id,name,ror,paper_id
283,2311.02036v1,"Université de Toulouse 3, Toulouse",02v6kpv12,2311.02036
5493,2311.15449v1,"Université Paul Sabatier, Toulouse",02v6kpv12,2311.15449
6987,2311.01163v1,"Université Toulouse III - Paul Sabatier, Toulouse",02v6kpv12,2311.01163
11853,2311.13894v1,"Université Toulouse 3, Toulouse",02v6kpv12,2311.13894
11988,2311.17019v1,"Université Paul Sabatier, Toulouse",02v6kpv12,2311.17019


In [67]:
#pd.DataFrame({'arx_id':list(scopus_arx_ids)}).to_csv("paris-saclay_arx_ids.csv", header=False)

In [40]:
named_inst = res_df[res_df['paper_id'].isin(scopus_ids)].value_counts('name', ascending=False)
named_inst.head()
res_df[res_df['paper_id'].isin(scopus_ids)].groupby(['name'])['ror'].value_counts(ascending=False).sort_values(ascending=False).head(20)
res_df[res_df['paper_id'].isin(scopus_ids)].head(10)
res_df[res_df['paper_id'].isin(scopus_ids) & res_df['name'].str.contains("Toulouse")].head(10)

name
Université de Toulouse, Toulouse                29
California Institute of Technology, Pasadena    15
ESO                                              9
NASA Goddard Space Flight Center, Greenbelt      9
IRAP, Toulouse                                   9
Name: count, dtype: int64

name                                          ror      
Université de Toulouse, Toulouse              017tgbk05    29
California Institute of Technology, Pasadena  05dxps055    15
ESO                                           00vvz3k68     9
NASA Goddard Space Flight Center, Greenbelt   0171mag52     9
Leiden University, Leiden                     027bh9e22     9
IRAP, Toulouse                                05hm2ja81     9
Université Paris-Saclay                       03xjwb503     8
California Institute of Technology            05dxps055     8
CNRS, Toulouse                                02feahw73     8
Institut d'Astrophysique de Paris, Paris      022bnxw24     8
Université Paris-Saclay, Gif-sur-Yvette       03xjwb503     8
University of Helsinki                        040af2s02     7
Univ. Grenoble Alpes                          03vte9x46     7
University of Manchester                      027m9bs27     7
University of Helsinki, Helsinki              040af2s02     7
Universidade d

,arx_id,name,ror,paper_id
282,2311.02036v1,"Université Lyon 1, Villeurbanne",029brtt94,2311.02036
283,2311.02036v1,"Université de Toulouse 3, Toulouse",02v6kpv12,2311.02036
284,2311.02036v1,"European Space Agency, Villanueva de la Cañada",03wd9za21,2311.02036
285,2311.02036v1,"Imperial College London, London",041kmwe10,2311.02036
661,2311.03288v1,"Université Paris Cité, Gif-sur-Yvette",05f82e368,2311.03288
662,2311.03288v1,"IRAP, Toulouse",05hm2ja81,2311.03288
663,2311.03288v1,"University of Leeds, Leeds",024mrxd33,2311.03288
751,2311.00034v1,"Northwestern University, Evanston",000e0be47,2311.00034
752,2311.00034v1,"Université de Toulouse, Toulouse",017tgbk05,2311.00034
753,2311.00034v1,"Harvard University, Cambridge",03vek6s52,2311.00034


,arx_id,name,ror,paper_id
283,2311.02036v1,"Université de Toulouse 3, Toulouse",02v6kpv12,2311.02036
662,2311.03288v1,"IRAP, Toulouse",05hm2ja81,2311.03288
752,2311.00034v1,"Université de Toulouse, Toulouse",017tgbk05,2311.00034
1391,2311.06450v3,"Institut de Mathématiqes de Toulouse, Toulouse",014vp6c30,2311.06450
1933,2311.07518v1,"LAAS, Toulouse",03vcm6439,2311.07518
4222,2311.05205v1,"Institut de Mathématiques de Toulouse, Toulouse",014vp6c30,2311.05205
4223,2311.05205v1,"Université de Toulouse, Toulouse",017tgbk05,2311.05205
4224,2311.05205v1,"CNRS, Toulouse",02feahw73,2311.05205
4225,2311.05205v1,"INSA, Toulouse",01h8pf755,2311.05205
4875,2311.14322v1,"Institut de Mathématiques de Toulouse, Toulouse",014vp6c30,2311.14322


In [41]:
paper_grps = res_df[res_df['paper_id'].isin(scopus_ids)].groupby(['arx_id'])['ror'].unique().reset_index()
sub_ids[:10]
error_df = paper_grps[paper_grps['ror'].apply(lambda x: all(i not in sub_ids for i in x))]
error_df.head(10)
check_ids = error_df['arx_id'].unique().tolist()

['04fhrs205',
 '003412r28',
 '02hbzmb19',
 '01225hq90',
 '034nb0f30',
 '045ktmd28',
 '05k0qmv73',
 '040smqw14',
 '025nmxp11',
 '016zvc994']

,arx_id,ror
0,2311.00034v1,"[000e0be47, 017tgbk05, 03vek6s52, 01zkghx44]"
1,2311.00433v2,"[012a77v79, 004raaa70, 05trd4x28]"
2,2311.00485v1,"[02x2v6p15, 012a91z28]"
4,2311.01300v2,"[03angcq70, 013meh722, 00vtgdb53, 047s2c258, 0..."
7,2311.01496v2,"[052gg0110, 01bf9rw71, 017tgbk05]"
8,2311.02015v2,"[0168r3w48, 013meh722, 02xp2c270]"
10,2311.02176v1,"[00gqsp710, 03c3r2d17, 00hx57361, 02p77k626, 0..."
12,2311.02507v3,[00240q980]
14,2311.03168v1,"[01swzsf04, 03ykbk197, 01111rn36, 01an7q238, 0..."
16,2311.03247v1,"[05n21n105, 04vmvtb21, 01rx4qw44]"


#### Kings College

First run :

 - 187 / 218 scopus papers contained Imperial College
 -  19 / 218 scopus papers contained King's College

Second run:

 - 232 / 265 scopus papers contained Imperial College
 -  19 / 265 scopus papers contained King's College

In [42]:


def pat_in_text(arx_id, cpat):
    yymm = arx_id.split(".")[0]
    paper_id = arx_id.split("v")[0]
    txt_path = f'txt/arxiv/{yymm}/{arx_id}.txt'
    client = storage.Client(project=PRD_PROJECT)
    bucket = client.bucket(PRD_BUCKET_LOC)
    blob = bucket.blob(txt_path)
    txt_bytes = blob.download_as_bytes()
    file_contents = txt_bytes.decode('utf-8')
    
    if cpat.search(file_contents):
        return True
    return False
    

In [43]:
cpat = re.compile(r"Toulouse") #re.compile(r"[Kk]ing'?s")
found = []
for arx_id in tqdm(check_ids):
    if pat_in_text(arx_id, cpat):
        found.append(arx_id)
len(found)

  0%|          | 0/67 [00:00<?, ?it/s]

65

In [43]:
cpat = re.compile(r"[Kk]ing['’]?s\s+College")
not_found = []
for arx_id in tqdm(check_ids):
    if not pat_in_text(arx_id, cpat):
        not_found.append(arx_id)
print(f"Found pat in {len(found)} of {len(check_ids)} papers.")

not_found

  0%|          | 0/24 [00:00<?, ?it/s]

Found pat in 16 of 24 papers.


['2311.05752v1',
 '2311.10443v2',
 '2311.13339v1',
 '2311.13835v1',
 '2311.14129v1',
 '2311.14177v1',
 '2311.17640v2',
 '2311.17640v3']

In [12]:
text = """
Lorem ipsum dolor sit amet, consectetur
adipiscing elit. Sed do eiusmod tempor
incididunt ut labore et dolore magna
aliqua. Ut enim ad minim veniam, quis
nostrud exercitation ullamco laboris nisi
ut aliquip ex ea commodo consequat. Kings
aute irure dolor in reprehenderit in
King's velit esse cillum dolore eu
fugiat nulla pariatur. Excepteur sint
occaecat cupidatat kings proident, sunt in
culpa qui officia deserunt mollit anim id
est laborum.
"""
pat = re.compile(r"[Kk]ing'?s")
pat.search(text)

<regex.Match object; span=(233, 238), match='Kings'>

In [204]:
all(i not in sub_ids for i in paper_grps['ror'][0])

True

In [44]:
arx_id = '2311.00034v1' #'2311.00018v1'
phase_one.get_single_file_results(arx_id, verbose=True)

[('2311.00034v1', 'Northwestern University', 'Evanston', '000e0be47'),
 ('2311.00034v1',
  'Institut de Recherche en Astrophysique et Planétologie',
  'Toulouse',
  '05hm2ja81'),
 ('2311.00034v1', 'Harvard University', 'Cambridge', '03vek6s52'),
 ('2311.00034v1', 'Georgia Institute of Technology', 'Atlanta', '01zkghx44')]

In [37]:
target_rors = ['02mp2av58']
res_df[res_df['paper_id'].isin(scopus_ids) & res_df['ror'].isin(target_rors)]

,arx_id,name,ror,paper_id
11587,2311.17969v1,Texas A&M University,02mp2av58,2311.17969
11588,2311.17969v1,New York University,02mp2av58,2311.17969
21437,2311.05877v1,New York University,02mp2av58,2311.05877
22638,2311.08970v1,University of Florida,02mp2av58,2311.08970
24791,2311.03534v2,Microsoft Research,02mp2av58,2311.03534
24792,2311.03534v2,Meta,02mp2av58,2311.03534
36274,2311.18494v1,Yandex LLC,02mp2av58,2311.18494
36276,2311.18494v1,New York University,02mp2av58,2311.18494
44177,2311.16098v1,Meta,02mp2av58,2311.16098
44191,2311.03386v1,New York University,02mp2av58,2311.03386


In [30]:
name_txt = "University of Colorado"
res_df[res_df['paper_id'].isin(scopus_ids) & res_df['name'].str.contains(name_txt)].head()

,arx_id,name,ror,paper_id
1931,2311.01187v1,"University of Colorado Boulder, Boulder",02ttsq026,2311.01187
6729,2311.18020v2,"University of Colorado Boulder, Boulder",02ttsq026,2311.18020
8553,2311.07483v1,"University of Colorado, Boulder",02ttsq026,2311.07483
8755,2311.06424v2,"University of Colorado Boulder,",02ttsq026,2311.06424
14750,2311.13322v2,"University of Colorado Boulder, Boulder",02ttsq026,2311.13322


## rerun small batch

In [120]:
importlib.reload(phase_one)

<module 'phase_one_json' from '/home/jupyter/metadata-vertexai/phase_one_json.py'>

In [122]:
#arx_id

In [9]:
test_id = '2311.00765v1'
phase_one.get_single_file_results(test_id, verbose=True, vverbose=True)

Processing ftp/arxiv/papers/2311/2311.00765.tar.gz
0: \author{Ramon Cardias}
\affiliation
{Department of Applied Physics, School of Engineering Sciences, KTH Royal Institute of Technology, AlbaNova University Center, SE-10691 Stockholm, Sweden}
\affiliation
{Instituto de Física, Universidade Federal Fluminense, 24210-346, Niterói RJ, Brazil}
\author{Simon Streib}
\affiliation
{Department of Physics and Astronomy, Uppsala University, Box 516,
SE-75120 Uppsala, Sweden}
\author{Zhiwei Lu}
\affiliation
{Department of Applied Physics, School of Engineering Sciences, KTH Royal Institute of Technology, AlbaNova University Center, SE-10691 Stockholm, Sweden}
\author{Manuel Pereiro }
\affiliation
{Department of Physics and Astronomy, Uppsala University, Box 516,
SE-75120 Uppsala, Sweden}
\author{Anders Bergman}
\affiliation
{Department of Physics and Astronomy, Uppsala University, Box 516,
SE-75120 Uppsala, Sweden}
\author{Erik Sj\"oqvist }
\affiliation
{Department of Physics and Astronomy, Upp

[('2311.00765v1',
  'KTH Royal Institute of Technology',
  'Stockholm',
  '026vcq606'),
 ('2311.00765v1', 'Universidade Federal Fluminense', 'Niterói', '02rjhbb08'),
 ('2311.00765v1', 'Uppsala University', 'Uppsala', '048a87296'),
 ('2311.00765v1', 'Université Paris-Saclay', 'Gif-sur-Yvette', '03xjwb503'),
 ('2311.00765v1', 'CEA', 'Gif-sur-Yvette', '00jjx8s55'),
 ('2311.00765v1', 'CNRS', 'Gif-sur-Yvette', '02feahw73'),
 ('2311.00765v1', 'SPEC', 'Gif-sur-Yvette', '03qsrw587'),
 ('2311.00765v1',
  'Swedish e-Science Research Center (SeRC)',
  'Stockholm',
  '01brr3227'),
 ('2311.00765v1',
  'Wallenberg Initiative Materials Science for Sustainability (WISE)',
  'Stockholm',
  'null'),
 ('2311.00765v1', 'Örebro University', 'Örebro', '05kytsw45')]

In [ ]:
len("Universit\u00e0")

In [72]:
king_df = pd.DataFrame.from_records(king_res_flat, columns=['arx_id', 'name', 'city', 'ror'])
king_df.to_csv("king-college.csv")
king_df.head()
processed_idx = set(king_df['arx_id'].unique())

,arx_id,name,city,ror
0,2311.17560v1,ic,,05wba8r86
1,2311.17560v1,sy,,047btv285
2,2311.17560v1,Imperial College London,,041kmwe10
3,2311.17560v1,University of Surrey,,00ks66431
4,2311.13070v1,University of Utah,Salt Lake City,03r0ha626


In [70]:
processed_idx = set()
king_res_flat = []

In [71]:
for arx_id in tqdm(scopus_arx_ids):
    if arx_id in processed_idx:
        continue
    res1 = phase_one.get_single_file_results(arx_id, verbose=False)
    king_res_flat.extend(res1)

  0%|          | 0/261 [00:00<?, ?it/s]

In [84]:
gemini_res = '''
```json
{"name": "Institute for Astronomy, University of Edinburgh", "city": "Edinburgh", "country": "UK"}
{"name": "Dipartimento di Scienze Matematiche, Fisiche e Informatiche, Universit\u00e0 di Parma", "city": "Parma", "country": "Italy"}
{"name": "Technion Israel Institute of Technology", "city": null, "country": "Israel"}
{"name": "SISSA, International School for Advanced Studies", "city": "Trieste", "country": "Italy"}
{"name": "ICSC - Centro Nazionale di Ricerca in High Performance Computing, Big Data e Quantum Computing", "city": "Bologna", "country": "Italy"}
{"name": "IFPU, Institute for Fundamental Physics of the Universe", "city": "Trieste", "country": "Italy"}
{"name": "INFN Gruppo Collegato di Parma", "city": "Parma", "country": "Italy"}
{"name": "Dipartimento di Fisica \"Aldo Pontremoli\", Universit\u00e0 degli Studi di Milano", "city": "Milano", "country": "Italy"}
{"name": "INAF-IASF Milano", "city": "Milano", "country": "Italy"}
{"name": "School of Physics and Astronomy, Queen Mary University of London", "city": "London", "country": "UK"}
{"name": "Institut de Physique Th\u00e9orique, CEA, CNRS, Universit\u00e9 Paris-Saclay", "city": "Gif-sur-Yvette Cedex", "country": "France"}
{"name": "Institute for Theoretical Particle Physics and Cosmology (TTK), RWTH Aachen University", "city": "Aachen", "country": "Germany"}
{"name": "Department of Physics \"E. Pancini\", University Federico II", "city": "Napoli", "country": "Italy"}
{"name": "Institute of Cosmology and Gravitation, University of Portsmouth", "city": "Portsmouth", "country": "UK"}
{"name": "Dipartimento di Fisica, Universit\u00e0 degli Studi di Torino", "city": "Torino", "country": "Italy"}
{"name": "INFN-Sezione di Torino", "city": "Torino", "country": "Italy"}
{"name": "INAF-Osservatorio Astrofisico di Torino", "city": "Pino Torinese", "country": "Italy"}
{"name": "Higgs Centre for Theoretical Physics, School of Physics and Astronomy, The University of Edinburgh", "city": "Edinburgh", "country": "UK"}
{"name": "Dipartimento di Fisica e Astronomia, Universit\u00e0 di Bologna", "city": "Bologna", "country": "Italy"}
{"name": "INAF-Osservatorio di Astrofisica e Scienza dello Spazio di Bologna", "city": "Bologna", "country": "Italy"}
{"name": "INFN-Sezione di Bologna", "city": "Bologna", "country": "Italy"}
{"name": "Istituto Nazionale di Fisica Nucleare, Sezione di Bologna", "city": "Bologna", "country": "Italy"}
{"name": "Universit\u00e9 de Gen\u00e8ve, D\u00e9partement de Physique Th\u00e9orique and Centre for Astroparticle Physics", "city": "Gen\u00e8ve", "country": "Switzerland"}
{"name": "Universit\u00e9 Paris-Saclay, CNRS, Institut d'astrophysique spatiale", "city": "Orsay", "country": "France"}
{"name": "School of Mathematics and Physics, University of Surrey", "city": "Guildford", "country": "UK"}
{"name": "INAF-Osservatorio Astronomico di Brera", "city": "Milano", "country": "Italy"}
{"name": "Max Planck Institute for Extraterrestrial Physics", "city": "Garching", "country": "Germany"}
{"name": "Dipartimento di Fisica, Universit\u00e0 di Genova", "city": "Genova", "country": "Italy"}
{"name": "INFN-Sezione di Genova", "city": "Genova", "country": "Italy"}
{"name": "INAF-Osservatorio Astronomico di Capodimonte", "city": "Napoli", "country": "Italy"}
{"name": "INFN section of Naples", "city": "Napoli", "country": "Italy"}
{"name": "Instituto de Astrof\u00edsica e Ci\u00eancias do Espa\u00e7o, Universidade do Porto, CAUP", "city": "Porto", "country": "Portugal"}
{"name": "INAF-Osservatorio Astronomico di Roma", "city": "Monteporzio Catone", "country": "Italy"}
{"name": "INFN-Sezione di Roma", "city": "Roma", "country": "Italy"}
{"name": "Institut de F\u00edsica d'Altes Energies (IFAE), The Barcelona Institute of Science and Technology", "city": "Bellaterra", "country": "Spain"}
{"name": "Port d'Informaci\u00f3 Cient\u00edfica", "city": "Bellaterra", "country": "Spain"}
{"name": "Dipartimento di Fisica e Astronomia \"Augusto Righi\" - Alma Mater Studiorum Universit\u00e0 di Bologna", "city": "Bologna", "country": "Italy"}
{"name": "Jodrell Bank Centre for Astrophysics, Department of Physics and Astronomy, University of Manchester", "city": "Manchester", "country": "UK"}
{"name": "European Space Agency/ESRIN", "city": "Frascati", "country": "Italy"}
{"name": "ESAC/ESA", "city": "Villanueva de la Ca\u00f1ada", "country": "Spain"}
{"name": "University of Lyon, Univ Claude Bernard Lyon 1, CNRS/IN2P3, IP2I Lyon", "city": "Villeurbanne", "country": "France"}
{"name": "Aix-Marseille Universit\u00e9, CNRS, CNES, LAM", "city": "Marseille", "country": "France"}
{"name": "Institute of Physics, Laboratory of Astrophysics, Ecole Polytechnique F\u00e9d\u00e9rale de Lausanne (EPFL), Observatoire de Sauverny", "city": "Versoix", "country": "Switzerland"}
{"name": "UCB Lyon 1, CNRS/IN2P3, IUF, IP2I Lyon", "city": "Villeurbanne", "country": "France"}
{"name": "Departamento de F\u00edsica, Faculdade de Ci\u00eancias, Universidade de Lisboa", "city": "Lisboa", "country": "Portugal"}
{"name": "Instituto de Astrof\u00edsica e Ci\u00eancias do Espa\u00e7o, Faculdade de Ci\u00eancias, Universidade de Lisboa", "city": "Lisboa", "country": "Portugal"}
{"name": "Department of Astronomy, University of Geneva", "city": "Versoix", "country": "Switzerland"}
{"name": "INAF-Istituto di Astrofisica e Planetologia Spaziali", "city": "Roma", "country": "Italy"}
{"name": "Department of Physics, Oxford University", "city": "Oxford", "country": "UK"}
{"name": "INFN-Padova", "city": "Padova", "country": "Italy"}
{"name": "Universit\u00e9 Paris-Saclay, Universit\u00e9 Paris Cit\u00e9, CEA, CNRS, AIM", "city": "Gif-sur-Yvette", "country": "France"}
{"name": "Institut d'Estudis Espacials de Catalunya (IEEC)", "city": "Barcelona", "country": "Spain"}
{"name": "Institut de Ciencies de l'Espai (IEEC-CSIC)", "city": "Barcelona", "country": "Spain"}
{"name": "INAF-Osservatorio Astronomico di Trieste", "city": "Trieste", "country": "Italy"}
{"name": "INAF-Osservatorio Astronomico di Padova", "city": "Padova", "country": "Italy"}
{"name": "University Observatory, Faculty of Physics, Ludwig-Maximilians-Universit\u00e4t", "city": "Munich", "country": "Germany"}
{"name": "INFN-Sezione di Milano", "city": "Milano", "country": "Italy"}
{"name": "Institute of Theoretical Astrophysics, University of Oslo", "city": "Oslo", "country": "Norway"}
{"name": "von Hoerner & Sulger GmbH", "city": "Schwetzingen", "country": "Germany"}
{"name": "Technical University of Denmark", "city": "Kgs. Lyngby", "country": "Denmark"}
{"name": "Cosmic Dawn Center (DAWN)", "city": null, "country": "Denmark"}
{"name": "Max-Planck-Institut f\u00fcr Astronomie", "city": "Heidelberg", "country": "Germany"}
{"name": "Department of Physics and Astronomy, University College London", "city": "London", "country": "UK"}
{"name": "Department of Physics and Helsinki Institute of Physics, Gustaf H\u00e4llstr\u00f6min katu 2, 00014 University of Helsinki", "city": "Helsinki", "country": "Finland"}
{"name": "Aix-Marseille Universit\u00e9, CNRS/IN2P3, CPPM", "city": "Marseille", "country": "France"}
{"name": "Jet Propulsion Laboratory, California Institute of Technology", "city": "Pasadena", "country": "USA"}
{"name": "AIM, CEA, CNRS, Universit\u00e9 Paris-Saclay, Universit\u00e9 de Paris", "city": "Gif-sur-Yvette", "country": "France"}
{"name": "Mullard Space Science Laboratory, University College London", "city": "Holmbury St Mary", "country": "UK"}
{"name": "Department of Physics, P.O. Box 64, 00014 University of Helsinki", "city": "Helsinki", "country": "Finland"}
{"name": "Helsinki Institute of Physics, Gustaf H\u00e4llstr\u00f6min katu 2, University of Helsinki", "city": "Helsinki", "country": "Finland"}
{"name": "NOVA optical infrared instrumentation group at ASTRON", "city": "Dwingeloo", "country": "The Netherlands"}
{"name": "Universit\u00e4t Bonn, Argelander-Institut f\u00fcr Astronomie", "city": "Bonn", "country": "Germany"}
{"name": "Dipartimento di Fisica e Astronomia \"Augusto Righi\" - Alma Mater Studiorum Universit\u00e0 di Bologna", "city": "Bologna", "country": "Italy"}
{"name": "Department of Physics, Institute for Computational Cosmology, Durham University", "city": "DH1 3LE", "country": "UK"}
{"name": "European Space Agency/ESTEC", "city": "Noordwijk", "country": "The Netherlands"}
{"name": "Department of Physics and Astronomy, University of Aarhus", "city": "Aarhus C", "country": "Denmark"}
{"name": "Centre for Astrophysics, University of Waterloo", "city": "Waterloo", "country": "Canada"}
{"name": "Department of Physics and Astronomy, University of Waterloo", "city": "Waterloo", "country": "Canada"}
{"name": "Perimeter Institute for Theoretical Physics", "city": "Waterloo", "country": "Canada"}
{"name": "Universit\u00e9 Paris-Saclay, Universit\u00e9 Paris Cit\u00e9, CEA, CNRS, Astrophysique, Instrumentation et Mod\u00e9lisation Paris-Saclay", "city": "Gif-sur-Yvette", "country": "France"}
{"name": "Space Science Data Center, Italian Space Agency", "city": "Roma", "country": "Italy"}
{"name": "Centre National d'Etudes Spatiales -- Centre spatial de Toulouse", "city": "Toulouse Cedex 9", "country": "France"}
{"name": "Institute of Space Science", "city": "M\u0103gurele", "country": "Romania"}
{"name": "Dipartimento di Fisica e Astronomia \"G. Galilei\", Universit\u00e0 di Padova", "city": "Padova", "country": "Italy"}
{"name": "Universit\u00e4ts-Sternwarte M\u00fcnchen, Fakult\u00e4t f\u00fcr Physik, Ludwig-Maximilians-Universit\u00e4t M\u00fcnchen", "city": "M\u00fcnchen", "country": "Germany"}
{"name": "Departamento de F\u00edsica, FCFM, Universidad de Chile", "city": "Santiago", "country": "Chile"}
{"name": "Institute of Space Sciences (ICE, CSIC)", "city": "Barcelona", "country": "Spain"}
{"name": "Satlantis", "city": "Leioa-Bilbao", "country": "Spain"}
{"name": "Centro de Investigaciones Energ\u00e9ticas, Medioambientales y Tecnol\u00f3gicas (CIEMAT)", "city": "Madrid", "country": "Spain"}
{"name": "Instituto de Astrof\u00edsica e Ci\u00eancias do Espa\u00e7o, Faculdade de Ci\u00eancias, Universidade de Lisboa", "city": "Lisboa", "country": "Portugal"}
{"name": "Universidad Polit\u00e9cnica de Cartagena, Departamento de Electr\u00f3nica y Tecnolog\u00eda de Computadoras", "city": "Cartagena", "country": "Spain"}
{"name": "Institut de Recherche en Astrophysique et Plan\u00e9tologie (IRAP), Universit\u00e9 de Toulouse, CNRS, UPS, CNES", "city": "Toulouse", "country": "France"}
{"name": "Kapteyn Astronomical Institute, University of Groningen", "city": "Groningen", "country": "The Netherlands"}
{"name": "INFN-Bologna", "city": "Bologna", "country": "Italy"}
{"name": "Infrared Processing and Analysis Center, California Institute of Technology", "city": "Pasadena", "country": "USA"}
{"name": "INAF, Istituto di Radioastronomia", "city": "Bologna", "country": "Italy"}
{"name": "Instituto de Astrof\u00edsica de Canarias", "city": "San Crist\u00f3bal de La Laguna", "country": "Spain"}
{"name": "Institut f\u00fcr Theoretische Physik, University of Heidelberg", "city": "Heidelberg", "country": "Germany"}
{"name": "Universit\u00e9 St Joseph; Faculty of Sciences", "city": "Beirut", "country": "Lebanon"}
{"name": "Institut d'Astrophysique de Paris", "city": "Paris", "country": "France"}
{"name": "Junia, EPA department", "city": "Lille", "country": "France"}
{"name": "INFN, Sezione di Trieste", "city": "Trieste TS", "country": "Italy"}
{"name": "Instituto de F\u00edsica Te\u00f3rica UAM-CSIC", "city": "Madrid", "country": "Spain"}
{"name": "CERCA/ISO, Department of Physics, Case Western Reserve University", "city": "Cleveland", "country": "USA"}
{"name": "Laboratoire Univers et Th\u00e9orie, Observatoire de Paris, Universit\u00e9 PSL, Universit\u00e9 Paris Cit\u00e9, CNRS", "city": "Meudon", "country": "France"}
{"name": "Dipartimento di Fisica e Scienze della Terra, Universit\u00e0 degli Studi di Ferrara", "city": "Ferrara", "country": "Italy"}
{"name": "Istituto Nazionale di Fisica Nucleare, Sezione di Ferrara", "city": "Ferrara", "country": "Italy"}
{"name": "Institut d'Astrophysique de Paris, UMR 7095, CNRS, and Sorbonne Universit\u00e9", "city": "Paris", "country": "France"}
{"name": "Dipartimento di Fisica - Sezione di Astronomia, Universit\u00e0 di Trieste", "city": "Trieste", "country": "Italy"}
{"name": "Minnesota Institute for Astrophysics, University of Minnesota", "city": "Minneapolis", "country": "USA"}
{"name": "Universit\u00e9 C\u00f4te d'Azur, Observatoire de la C\u00f4te d'Azur, CNRS, Laboratoire Lagrange", "city": "Nice cedex 4", "country": "France"}
{"name": "Institute Lorentz, Leiden University", "city": "Leiden", "country": "The Netherlands"}
{"name": "Institute for Astronomy, University of Hawaii", "city": "Honolulu", "country": "USA"}
{"name": "Department of Physics & Astronomy, University of California Irvine", "city": "Irvine", "country": "USA"}
{"name": "Department of Astronomy & Physics and Institute for Computational Astrophysics, Saint Mary's University", "city": "Halifax", "country": "Canada"}
{"name": "Departamento F\u00edsica Aplicada, Universidad Polit\u00e9cnica de Cartagena", "city": "Cartagena", "country": "Spain"}
{"name": "Universit\u00e9 Paris Cit\u00e9, CNRS, Astroparticule et Cosmologie", "city": "Paris", "country": "France"}
{"name": "Department of Computer Science, Aalto University", "city": "Espoo", "country": "Finland"}
{"name": "Department of Physics and Astronomy, Vesilinnantie 5, 20014 University of Turku", "city": "Turku", "country": "Finland"}
{"name": "Serco for European Space Agency (ESA)", "city": "Villanueva de la Ca\u00f1ada", "country": "Spain"}
{"name": "ARC Centre of Excellence for Dark Matter Particle Physics", "city": "Melbourne", "country": "Australia"}
{"name": "Centre for Astrophysics & Supercomputing, Swinburne University of Technology", "city": "Victoria", "country": "Australia"}
{"name": "W.M. Keck Observatory", "city": "Kamuela", "country": "USA"}
{"name": "Department of Physics and Astronomy, University of the Western Cape", "city": "Bellville", "country": "South Africa"}
{"name": "Oskar Klein Centre for Cosmoparticle Physics, Department of Physics, Stockholm University", "city": "Stockholm", "country": "Sweden"}
{"name": "Astrophysics Group, Blackett Laboratory, Imperial College London", "city": "London", "country": "UK"}
{"name": "Univ. Grenoble Alpes, CNRS, Grenoble INP, LPSC-IN2P3", "city": "Grenoble", "country": "France"}
{"name": "Dipartimento di Fisica, Sapienza Universit\u00e0 di Roma", "city": "Roma", "country": "Italy"}
{"name": "Centro de Astrof\u00edsica da Universidade do Porto", "city": "Porto", "country": "Portugal"}
{"name": "Zentrum f\u00fcr Astronomie, Universit\u00e4t Heidelberg", "city": "Heidelberg", "country": "Germany"}
{"name": "Dipartimento di Fisica, Universit\u00e0 di Roma Tor Vergata", "city": "Roma", "country": "Italy"}
{"name": "INFN, Sezione di Roma 2", "city": "Roma", "country": "Italy"}
{"name": "Institute of Astronomy, University of Cambridge", "city": "Cambridge", "country": "UK"}
{"name": "Institute for Computational Science, University of Zurich", "city": "Zurich", "country": "Switzerland"}
{"name": "Department of Astrophysical Sciences, Peyton Hall, Princeton University", "city": "Princeton", "country": "USA"}
{"name": "Niels Bohr Institute, University of Copenhagen", "city": "Copenhagen", "country": "Denmark"}
```
'''
found_institutions = []
verbose=True
for row in gemini_res.strip().splitlines():
    if row:
        if row.startswith('`'):
            continue
        try:
            found_institutions.append(json.loads(r'{}'.format(row)))
        except json.JSONDecodeError as e:
            if verbose:
                print(f"JSONDecodeError: {e} on {arx_id} at {row}")
            pass

JSONDecodeError: Expecting ',' delimiter: line 1 column 35 (char 34) on 2311.13529v2 at {"name": "Dipartimento di Fisica "Aldo Pontremoli", Università degli Studi di Milano", "city": "Milano", "country": "Italy"}
JSONDecodeError: Expecting ',' delimiter: line 1 column 34 (char 33) on 2311.13529v2 at {"name": "Department of Physics "E. Pancini", University Federico II", "city": "Napoli", "country": "Italy"}
JSONDecodeError: Expecting ',' delimiter: line 1 column 48 (char 47) on 2311.13529v2 at {"name": "Dipartimento di Fisica e Astronomia "Augusto Righi" - Alma Mater Studiorum Università di Bologna", "city": "Bologna", "country": "Italy"}
JSONDecodeError: Expecting ',' delimiter: line 1 column 48 (char 47) on 2311.13529v2 at {"name": "Dipartimento di Fisica e Astronomia "Augusto Righi" - Alma Mater Studiorum Università di Bologna", "city": "Bologna", "country": "Italy"}
JSONDecodeError: Expecting ',' delimiter: line 1 column 48 (char 47) on 2311.13529v2 at {"name": "Dipartimento di Fisi

In [123]:
gemini_res = '''
{"name": "Universit\\`a di Padova", "city": "Padova", "country": "Italy"}
{"name": "Department of Physics \\"E. Pancini\\", University Federico II", "city": "Napoli", "country": "Italy"}
'''
institutions_found = []
verbose=True
for row in gemini_res.strip().splitlines():
    if row:
        if row.startswith('`'):
            continue
        try:
            institutions_found.append(json.loads(row))
        except json.JSONDecodeError as e:
            try:
                institutions_found.append(json.loads(r"{}".format(row).replace('\\', '\\\\')))
            except json.JSONDecodeError:
                if verbose:
                    print(f"JSONDecodeError: {e} on {arx_id} at {row}")
                pass
institutions_found

[{'name': 'Universit\\`a di Padova', 'city': 'Padova', 'country': 'Italy'},
 {'name': 'Department of Physics "E. Pancini", University Federico II',
  'city': 'Napoli',
  'country': 'Italy'}]

## Scratch

In [ ]:
ror
(ror in skip_inst)
ror_df[ror_df['ror']==ror]
ror_map_df[ror_map_df['ror']==ror]

In [ ]:
ror_df.loc[ror_df['ror']==ror,'name'].iloc[0]

In [ ]:
res = {
    "ror": ror,
    "TP": len(res_ids.intersection(scopus_ids)), 
    "FP": len(res_ids - scopus_ids), 
    "FN": len(scopus_ids - res_ids), 
    "TN": len((scopus_all - scopus_ids) - res_ids)
}

In [ ]:
list(itr.islice(scopus_ids, 10))

In [ ]:
list(itr.islice(res_ids, 10))

In [ ]:
#%%time
#res_dict = {}
#for arx_id in tqdm(false_positive): #scopus_positive:
#    #print(arx_id)
#    paper_id = arx_id.split("v")[0]
#    res = phase_one.send_one_submission_to_gemini(arx_id)
#    #print(f"\n{paper_id}\n{res}")
#    res_dict[paper_id] = res
#

### Phase 2 Name --> ROR id

Based on FAISS

In [83]:
importlib.reload(phase_one)

<module 'phase_one_json' from '/home/jupyter/metadata-vertexai/phase_one_json.py'>

In [9]:
ror_finder = phase_one.ROR_FINDER

In [97]:
ror_finder20 = phase_one.rorFinder(doc_k=20)

In [10]:
ror_finder.qa_chain.invoke({"query": 'Université Paris-Saclay'})

{'query': 'Université Paris-Saclay',
 'result': '03xjwb503\n',
 'source_documents': [Document(id='7918d35f-b965-45e1-a27f-4cd494092528', metadata={}, page_content='Université Paris-Saclay — https://ror.org/03xjwb503'),
  Document(id='095956bf-36bc-461c-9846-7e25588f9325', metadata={}, page_content='University of Paris-Saclay — https://ror.org/03xjwb503'),
  Document(id='8b55481d-f7c0-4041-b7c8-82bbbd7656e6', metadata={}, page_content='Universitat París-Saclay — https://ror.org/03xjwb503'),
  Document(id='787402cb-f2bb-497a-97ee-0fabc508f413', metadata={}, page_content='Université Paris III, Paris — https://ror.org/03z6jp965'),
  Document(id='538dae26-f091-4614-874d-50aaaf320d63', metadata={}, page_content='OSUPS - Université Paris-Saclay — https://ror.org/02b6c0m75')]}

In [99]:
ror_finder20.qa_chain.invoke({"query": 'Université Paris-Saclay, Gif-sur-Yvette'})

{'query': 'Université Paris-Saclay, Gif-sur-Yvette',
 'result': '03xjwb503\n',
 'source_documents': [Document(id='5c27f3ec-12dc-4abf-a961-74c855847901', metadata={}, page_content='Université Paris-Saclay, Gif-sur-Yvette — https://ror.org/03xjwb503'),
  Document(id='1a5b13ff-31ad-44e3-84c9-35f73954e741', metadata={}, page_content='University of Paris-Saclay, Gif-sur-Yvette — https://ror.org/03xjwb503'),
  Document(id='9a6e6d3e-500e-4769-b22f-76290d47c594', metadata={}, page_content='Universitat París-Saclay, Gif-sur-Yvette — https://ror.org/03xjwb503'),
  Document(id='a6063f78-43e9-4f27-a3f6-b1e619fa4b18', metadata={}, page_content='École Normale Supérieure Paris-Saclay, Gif-sur-Yvette — https://ror.org/00hx6zz33'),
  Document(id='fa7eb3f4-2c67-4469-839f-d6645d05a02f', metadata={}, page_content='CEA Paris-Saclay, Gif-sur-Yvette — https://ror.org/03n15ch10'),
  Document(id='4b13cad6-2777-4db9-89e6-b8e70111983f', metadata={}, page_content="Institut de Recherche sur les Lois Fondamentales 

In [209]:
ror_finder5.qa_chain.invoke({"query": 'New York University, New York'})

{'query': 'New York University, New York',
 'result': '0190ak572\n',
 'source_documents': [Document(id='6e603100-4e7c-447c-8e6c-d9f01ab5aa5f', metadata={}, page_content='York College, City University of New York, New York — https://ror.org/015a1ak54'),
  Document(id='a938e6e9-78ca-4711-9b10-485ed79801ff', metadata={}, page_content='City University of New York, New York — https://ror.org/00453a208'),
  Document(id='5cdb0189-0b74-46c8-89e7-f40fb3e3b4d0', metadata={}, page_content='New York University, New York — https://ror.org/0190ak572'),
  Document(id='5368175c-fbb5-4dfc-957b-e8840930398c', metadata={}, page_content='York University, York — https://ror.org/022jz8688'),
  Document(id='d07f8630-c036-4e23-8095-4a3fc3ca86d8', metadata={}, page_content='University of York, York — https://ror.org/04m01e293')]}

In [212]:
%%time
ror_finder5.get_ror('New York University, New York')

CPU times: user 8 µs, sys: 1e+03 ns, total: 9 µs
Wall time: 13.4 µs


'0190ak572'

In [ ]:
%%time
ror_finder.get_ror('Universitas Indonesia,')

## Prompt Experiments

In [67]:
input_text = '''
 \author{
    Sonish Sivarajkumar, MS\textsuperscript{1}\textsuperscript{,2}\thanks{Present address: School of Computing and Information, University of Pittsburgh, Pennsylvania, PA, USA. Work was done while at Molecular Robotics, Kerala, India.},
    Pratyush Tandale, MS\textsuperscript{3}, 
    Ankit Bhardwaj, BS\textsuperscript{4}, \\
    Kipp W. Johnson, MD,PhD\textsuperscript{5}, 
    Anoop Titus, MD\textsuperscript{6}, 
    Benjamin S. Glicksberg, PhD\textsuperscript{7},\\
    Shameer Khader, PhD, MPH\textsuperscript{8}\textsuperscript{\dag}, 
    Kamlesh K. Yadav, PhD\textsuperscript{9, 10}\textsuperscript{\dag}, \\
    Lakshminarayanan Subramanian, PhD\textsuperscript{4}\thanks{Corresponding authors: shameer.khader20@imperial.ac.uk, kamlesh.yadav@tamu.edu, lakshmi@cs.nyu.edu}
}
,
    Pratyush Tandale, MS
,
    Pratyush Tandale, MS
, 
    Ankit Bhardwaj, BS
, 
, 
    Anoop Titus, MD
, 
    Benjamin S. Glicksberg, PhD
,
, 
    Kamlesh K. Yadav, PhD
, 
    Kamlesh K. Yadav, PhD
, 
, 


Corresponding authors
Molecular Robotics, Kerala, India; 
School of Computing and Information, University of Pittsburgh, Pennsylvania, PA, USA; 
Health Informatics  &  Data Science, Georgetown University, Washington DC, USA; 
Department of Computer Science, Courant Institute of Mathematical Sciences, New York University, New York, NY, USA; 
Institute for Next Generation Healthcare, Mount Sinai Health System, New York, NY, USA; 
Department of Preventive Cardiology, DeBakey Heart  &  Vascular Center, Houston Methodist, Houston, TX, USA; 
Hasso Plattner Institute for Digital Health, Icahn School of Medicine at Mount Sinai, New York, NY, USA; 
Faculty of Medicine, Imperial College London, London, UK; 
School of Engineering Medicine,  Texas A & M University, Houston, TX, USA; 
Department of Translational Medical Sciences, Center for Genomic and Precision Medicine, Texas A & M University, Houston, TX, USA; 
'''.strip()

input_text = r'''
\author{Pascal Auscher}
\address
{Universit{\'e} Paris-Saclay, CNRS, Laboratoire de Math\'{e}matiques d'Orsay, 91405 Orsay, France}
\author{Hedong Hou}
\address
{Universit{\'e} Paris-Saclay, CNRS, Laboratoire de Math\'{e}matiques d'Orsay, 91405 Orsay, France}
'''.strip()

input_text3 ='''
Universit\'e Paris-Saclay, Universit\'e Paris Cit\'e, CEA, CNRS, Astrophysique, Instrumentation et Mod\'elisation Paris-Saclay, 91191 Gif-sur-Yvette, France\label{aff80}
\and
Space Science Data Center, Italian Space Agency, via del Politecnico snc, 00133 Roma, Italy\label{aff81}
\and
Centre National d'Etudes Spatiales -- Centre spatial de Toulouse, 18 avenue Edouard Belin, 31401 Toulouse Cedex 9, France\label{aff82}
\and
Dipartimento di Fisica e Astronomia "G. Galilei", Universit\`a di Padova, Via Marzolo 8, 35131 Padova, Italy\label{aff84}
\and
Universit\"ats-Sternwarte M\"unchen, Fakult\"at f\"ur Physik, Ludwig-Maximilians-Universit\"at M\"unchen, Scheinerstrasse 1, 81679 M\"unchen, Germany\label{aff85}
\and
Departamento de F\'isica, FCFM, Universidad de Chile, Blanco Encalada 2008, Santiago, Chile\label{aff86}
\and
Institute of Space Sciences (ICE, CSIC), Campus UAB, Carrer de Can Magrans, s/n, 08193 Barcelona, Spain\label{aff87}
\and
Satlantis, University Science Park, Sede Bld 48940, Leioa-Bilbao, Spain\label{aff88}
\and
Centro de Investigaciones Energ\'eticas, Medioambientales y Tecnol\'ogicas (CIEMAT), Avenida Complutense 40, 28040 Madrid, Spain\label{aff89}
\and
Instituto de Astrof\'isica e Ci\^encias do Espa\c{c}o, Faculdade de Ci\^encias, Universidade de Lisboa, Tapada da Ajuda, 1349-018 Lisboa, Portugal\label{aff90}
'''

inst_list = '''
University of Pittsburgh, Pennsylvania
Georgetown University, Washington DC
New York University, New York
Mount Sinai Health System, New York
Houston Methodist, Houston
Icahn School of Medicine at Mount Sinai, New York
Imperial College London, London
Texas A & M University, Houston
'''.strip()



VERIFY_TEMPLATE = """
Match the institution names in the LIST_OF_NAMES with the contents of the SOURCE_TEXT.
Answer "True" if ALL the institutions in the LIST_OF_NAMES are present in the SOURCE_TEXT, otherwise answer "False"\n
Only respond with "True" or "False".
### LIST_OF_NAMES:\n
{inst_list}\n\n

### SOURCE_TEXT:\n
{source_text}\n\n
""".strip()

VERIFY_TEMPLATE = """
Find all the organizations and any associated locations mentioned in the SOURCE_TEXT.
When organizations are listed together at an address, assume the location information ONLY applies to the immediately preceeding organization.

2. Output Format:
    - Each organization and location, if any, should be on a separate line with NO extra numbering, punctuation, or bullet points.
    - Do NOT include explanations, descriptions, or any other text — ONLY the organization list or 'null'
    - The output should be plain text with no markdown
    - Deduplicate the organization list 
    - Follow this pseudocode to generate the output:
    ```
    if no organization are found, then output "null".
    else
        for each organization
            if the organization is associated with a location
                if the location includes a city and country:
                    output a json object with this format: {{"name":organization, "city":city , "country":country}}
                else:
                   output a json object with this format: {{"name':organization, "city":city}}
            else 
                output a json object with this format: {{"name":organization}}
    ```

### SOURCE_TEXT:\n
{source_text}\n\n
""".strip()

foo = '''
___STRICT OUTPUT REQUIREMENTS:
Output Format:
    - Each organization should be on a separate line with NO extra numbering, punctuation, or bullet points.
    - Do NOT include explanations, descriptions, or any other text — ONLY the organization list or 'null'
    - The output should be plain text with no markdown
    - Deduplicate the organization list 
    - Follow this pseudocode to generate the output:

    if no organization are found, then output "null".
    else
        for each organization
            
            if organization is associated with a city
                if the city includes a country:
                    output a json object with this format: {{"name":organization, "city":city , "country":country}}
                else:
                   output a json object with this format: {{"name':organization, "city":city}}
            else 
                output a json object with this format: {{"name":organization}}
'''


VERIFY_TEMPLATE2 = """
You are an expert in recognizing organization names and associated locations in text and latex input.

Identify the authors' organizations in the following INPUT TEXT below.
When organizations are listed together at an address, assume the location information ONLY applies to the immediately preceeding organization.

___INPUT_TEXT:\n
{source_text}\n\n
""".strip()

foo = """
### OUTPUT FORMAT:
 - Convert any LaTeX character macros to utf-8.
 - print the number of organizations found
 - List each organization and its location as a separate item.
 - If the organization is not associated with a location in the SOURCE_TEXT, do not include location information for that organization

 - print the number of organizations found
 - explain your reasoning

"""

VERIFY_TEMPLATE = """
Output a list of organizations from the SOURCE_TEXT as descibed in the OUTPUT_FORMAT directions.
Follow the STEPS below:

### REQUIREMENTS
 - Consider each organization separately.
 - Convert any LaTeX character macros to utf-8.

### STEP 1:
 - Find all potential organizations AND, if present, any associated locations in the SOURCE_TEXT.
 - When organizations are listed together at an address, treat each organization as a separate entity.
 - When organizations are listed together at an address, assume the location information ONLY applies to the immediately preceeding organization.

### OUTPUT_FORMAT:
 - print the number of organizations found
 - List each organization and its location as a separate item.
 - 
 
### SOURCE_TEXT:\n
{source_text}\n\n
""".strip()





VERIFY_TEMPLATE = """
TASK: Follow the directions to generate output from the SOURCE_TEXT as descibed in the OUTPUT_FORMAT directions.
Follow the directions below:

 - Find all potential organizations AND, if present, any associated locations in the SOURCE_TEXT.
 - When organizations are listed together at an address, treat each organization as a separate entity.
 - When organizations are listed together at an address, assume the location information ONLY applies to the last organization in the list.

### OUTPUT_FORMAT:
 - render any LaTeX in organization names to unicode
 - The output should be plain text
 - Do not use markdown
 - Follow this pseudocode to generate the output:
```
    if no organizations are found, then output "null".
    else
        for each organization
            let org_name = name of the organization
            if organization is associated with a city
                if the city includes a country:
                    output a string with this format: {{"name":org_name, "city":city , "country":country}}
                else:
                   output a string with this format: {{"name':org_name, "city":city}}
            else 
                output a string with this format: {{"name":org_name}}
```

### SOURCE_TEXT:\n
{source_text}\n\n
""".strip()

input_text3 = '''
\author{Ramon Cardias}
\affiliation
{Department of Applied Physics, School of Engineering Sciences, KTH Royal Institute of Technology, AlbaNova University Center, SE-10691 Stockholm, Sweden}
\affiliation
{Instituto de Física, Universidade Federal Fluminense, 24210-346, Niterói RJ, Brazil}
\author{Simon Streib}
\affiliation
{Department of Physics and Astronomy, Uppsala University, Box 516,
SE-75120 Uppsala, Sweden}
\author{Zhiwei Lu}
\affiliation
{Department of Applied Physics, School of Engineering Sciences, KTH Royal Institute of Technology, AlbaNova University Center, SE-10691 Stockholm, Sweden}
\author{Manuel Pereiro }
\affiliation
{Department of Physics and Astronomy, Uppsala University, Box 516,
SE-75120 Uppsala, Sweden}
\author{Anders Bergman}
\affiliation
{Department of Physics and Astronomy, Uppsala University, Box 516,
SE-75120 Uppsala, Sweden}
\author{Erik Sj\"oqvist }
\affiliation
{Department of Physics and Astronomy, Uppsala University, Box 516,
SE-75120 Uppsala, Sweden}
\author{Cyrille Barreteau}
\affiliation
{Universit\'e Paris-Saclay, CEA, CNRS, SPEC, 91191, Gif-sur-Yvette, France}
\author{Anna Delin }
\affiliation
{Department of Applied Physics, School of Engineering Sciences, KTH Royal Institute of Technology, AlbaNova University Center, SE-10691 Stockholm, Sweden}
\affiliation
{ Swedish e-Science Research Center (SeRC), KTH Royal Institute of Technology, SE-10044 Stockholm, Sweden}
\affiliation
{Wallenberg Initiative Materials Science for Sustainability (WISE), KTH Royal Institute of Technology, SE-10044 Stockholm, Sweden}
\author{Olle Eriksson }
\affiliation
{Department of Physics and Astronomy, Uppsala University, Box 516,
SE-75120 Uppsala, Sweden}
\affiliation
{Wallenberg Initiative Materials Science for Sustainability (WISE), Uppsala University, Box 516,
SE-75120 Uppsala, Sweden}
\author{Danny Thonig}
\affiliation
{School of Science and Technology, \"Orebro University, SE-70182 Örebro,
Sweden}
\affiliation
{Department of Physics and Astronomy, Uppsala University, Box 516,
SE-75120 Uppsala, Sweden}
'''
input_text3 = '''
\title[Large-scale Magnetorotational Dynamo]{Magnetorotational dynamo can generate large-scale vertical magnetic fields in 3D GRMHD simulations of accreting black holes}



\author[J. Jacquemin-Ide et al.]{
Jonatan Jacquemin-Ide,$^{1}$\thanks{E-mail: jonatan.jacqueminide@northwestern.edu}
François Rincon, $^{2}$
Alexander Tchekhovskoy,$^{1}$
and Matthew Liska$^{3,4}$
\\

$^{1}$Center for Interdisciplinary Exploration $\&$ Research in Astrophysics (CIERA), Physics and Astronomy, Northwestern University, Evanston, IL 60202, USA\\
$^{2}$Institut de Recherche en Astrophysique et Planétologie (IRAP), Université de Toulouse, CNRS, UPS, Toulouse, France\\
$^{3}$Institute for Theory and Computation, Harvard University, 60 Garden Street, Cambridge, MA 02138, USA\\
$^{4}$Center for Relativistic Astrophysics, Georgia Institute of Technology, Howey Physics Bldg, 837 State St NW, Atlanta, GA 30332, USA\\
}


\date{Accepted XXX. Received YYY; in original form ZZZ}


\pubyear{2015}


\begin{document}
'''

input_text4 = '''
\author[J. Jacquemin-Ide et al.]{
Jonatan Jacquemin-Ide,$^{1}$\thanks{E-mail: jonatan.jacqueminide@northwestern.edu}
François Rincon, $^{2}$
Alexander Tchekhovskoy,$^{1}$
and Matthew Liska$^{3,4}$
\\
$^{1}$Center for Interdisciplinary Exploration $\&$ Research in Astrophysics (CIERA), Physics and Astronomy, Northwestern University, Evanston, IL 60202, USA\\
$^{2}$Institut de Recherche en Astrophysique et Planétologie (IRAP), Université de Toulouse, CNRS, UPS, Toulouse, France\\
$^{3}$Institute for Theory and Computation, Harvard University, 60 Garden Street, Cambridge, MA 02138, USA\\
$^{4}$Center for Relativistic Astrophysics, Georgia Institute of Technology, Howey Physics Bldg, 837 State St NW, Atlanta, GA 30332, USA\\
}


\date{Accepted XXX. Received YYY; in original form ZZZ}


\pubyear{2015}


\begin{document}
'''

VERIFY_TEMPLATE = """
TASK: Follow the directions to generate output from the SOURCE_TEXT as descibed in the OUTPUT_FORMAT directions.
Follow the directions below:
 - Find all potential organizations in the SOURCE_TEXT.
 - Expand abbreviations and acronyms of potential organization names.
 - When organizations are listed together at an address, include each organization as a separate item.
 - When organizations are listed together at an address, include any acronyms as a separate entity.
 - Identify any locations associated explicity associated with any of the potential organizations.
 - When organizations are listed together at a single address, ONLY associate the address with the last organization in the list.

### OUTPUT_FORMAT:
 - The output should be valid utf-8 line json
 - Output one json Object per line.
 - Do not return a json Array.
 - Replace any latex escape sequences in the output with utf-8 characters
 - double-escape all backslashes
 - Only report the main organizations like universities, universi, commissions, foundations or corporations.
 - Ignore sub-units like department, dipartimento, or college.
 - Follow this pseudocode to generate the output:
```
    if no organizations are found, then output "null".
    else
        for each organization
            let org_name be the organization name.
            let city be "" unless you identied a city for this organization
            let country be "" unless you identified a country location for this organization
            output a json Object with this format: {{"name":org_name, "city":city , "country":country}}
```
### SOURCE_TEXT:
{source_text}\n\n
""".strip()


res = phase_one.verify_with_gemini_api(inst_list, input_text4, template=VERIFY_TEMPLATE)
print(res)
for row in res.strip().splitlines():
    if row:
        try:
            j = json.loads(row)
            if j is None:
                print("is None")
            else:
                print(json.loads(row))
        except json.JSONDecodeError as e:
            try:
                print(json.loads(r"{}".format(row)))
            except:
                print(f"Error: {repr(row)}")
            pass

```json
{"name": "Northwestern University", "city": "Evanston", "country": "USA"}
{"name": "Université de Toulouse", "city": "Toulouse", "country": "France"}
{"name": "Harvard University", "city": "Cambridge", "country": "USA"}
{"name": "Georgia Institute of Technology", "city": "Atlanta", "country": "USA"}
```

Error: '```json'
{'name': 'Northwestern University', 'city': 'Evanston', 'country': 'USA'}
{'name': 'Université de Toulouse', 'city': 'Toulouse', 'country': 'France'}
{'name': 'Harvard University', 'city': 'Cambridge', 'country': 'USA'}
{'name': 'Georgia Institute of Technology', 'city': 'Atlanta', 'country': 'USA'}
Error: '```'


In [62]:
input_text3 = '''
$^{1}$Center for Interdisciplinary Exploration $\&$ Research in Astrophysics (CIERA), Physics and Astronomy, Northwestern University, Evanston, IL 60202, USA\\
$^{2}$Institut de Recherche en Astrophysique et Planétologie (IRAP), Université de Toulouse, CNRS, UPS, Toulouse, France\\
$^{3}$Institute for Theory and Computation, Harvard University, 60 Garden Street, Cambridge, MA 02138, USA\\
$^{4}$Center for Relativistic Astrophysics, Georgia Institute of Technology, Howey Physics Bldg, 837 State St NW, Atlanta, GA 30332, USA\\
'''
VERIFY_TEMPLATE = """
TASK: Follow the directions to generate output from the SOURCE_TEXT as descibed in the OUTPUT_FORMAT directions.
Follow the directions below:
 - Find all potential organizations in the SOURCE_TEXT.
 - Expand abbreviations and acronyms of potential organization names.
 - When organizations are listed together at an address, treat each organization as a separate entity.
 - When organizations are listed together at an address, expand any acronyms as a separate entity.
 - Identify any locations associated explicity associated with any of the potential organizations.
 - When organizations are listed together at a single address, ONLY associate the address with the last organization in the list.
 - Ignore any sub-units like departments or colleges.

### OUTPUT_FORMAT:
 - The output should be valid utf-8 line json
 - Output one json Object per line.
 - Do not return a json Array.
 - Replace any latex escape sequences in the output with utf-8 characters
 - double-escape all backslashes
 - Only report the main organizations like universities, universi, commissions, foundations or corporations.
 - Ignore sub-units like department, dipartimento, or college.
 - Follow this pseudocode to generate the output:
```
    if no organizations are found, then output "null".
    else
        for each organization
            let org_name be the organization name.
            let city be "" unless you identied a city for this organization
            let country be "" unless you identified a country location for this organization
            output a json Object with this format: {{"name":org_name, "city":city , "country":country}}

### SOURCE_TEXT:
{source_text}\n\n
""".strip()

VERIFY_TEMPLATE2 = '''
identify the organizations mentioned in the SOURCE_TEXT.
list the full name of the organizations

### SOURCE_TEXT:
{source_text}\n\n
'''

res = phase_one.verify_with_gemini_api(inst_list, input_text3, template=VERIFY_TEMPLATE)
print(res)
for row in res.strip().splitlines():
    if row:
        try:
            j = json.loads(row)
            if j is None:
                print("is None")
            else:
                print(json.loads(row))
        except json.JSONDecodeError as e:
            try:
                print(json.loads(r"{}".format(row)))
            except:
                print(f"Error: {repr(row)}")
            pass

```json
{"name": "Center for Interdisciplinary Exploration and Research in Astrophysics", "city": "Evanston", "country": "USA"}
{"name": "Northwestern University", "city": "Evanston", "country": "USA"}
{"name": "Institut de Recherche en Astrophysique et Planétologie", "city": "Toulouse", "country": "France"}
{"name": "Université de Toulouse", "city": "Toulouse", "country": "France"}
{"name": "CNRS", "city": "Toulouse", "country": "France"}
{"name": "UPS", "city": "Toulouse", "country": "France"}
{"name": "Institute for Theory and Computation", "city": "Cambridge", "country": "USA"}
{"name": "Harvard University", "city": "Cambridge", "country": "USA"}
{"name": "Center for Relativistic Astrophysics", "city": "Atlanta", "country": "USA"}
{"name": "Georgia Institute of Technology", "city": "Atlanta", "country": "USA"}
```

Error: '```json'
{'name': 'Center for Interdisciplinary Exploration and Research in Astrophysics', 'city': 'Evanston', 'country': 'USA'}
{'name': 'Northwestern Universit

In [56]:
            let org_name be the result of rendering any LaTeX in the organization name string to utf-8


SyntaxError: invalid syntax (1278732566.py, line 1)

In [84]:
arx_id = '2311.04844v2'
old_template = phase_one.PROMPT_TEMPLATE
phase_one.PROMPT_TEMPLATE = """
TASK: Follow the directions to generate output from the SOURCE_TEXT as descibed in the OUTPUT_FORMAT directions.
Follow the directions below:
 - Find all potential organizations in the SOURCE_TEXT.
 - When organizations are listed together at an address, treat each organization as a separate entity.
 - Identify any locations associated explicity associated with any of the potential organizations.
 - When organizations are listed together at a single address, ONLY associate the address with the last organization in the list.

### OUTPUT_FORMAT:
 - The output should be valid utf-8 line json
 - replace any latex characters in the output with utf-8 equivalent
 - Do not use markdown
 - Do not enclose output in backticks
 - Follow this pseudocode to generate the output:
```
    if no organizations are found, then output "null".
    else
        for each organization
            let org_name be the organization name
            let city be "" unless you identied a city for this organization
            let country be "" unless you identified a country location for this organization
            output a string with this format: {{"name":org_name, "city":city , "country":country}}

### SOURCE_TEXT:\n
{input_text}\n\n
""".strip()
res = phase_one.get_single_file_results(arx_id, verbose=True)
phase_one.PROMPT_TEMPLATE = old_template

res
#print(json.loads(res[0][1]))


Processing ftp/arxiv/papers/2311/2311.04844.tar.gz
0: \author{Pascal Auscher}
\address
{Universit{\'e} Paris-Saclay, CNRS, Laboratoire de Math\'{e}matiques d'Orsay, 91405 Orsay, France}
\author{Hedong Hou}
\address
{Universit{\'e} Paris-Saclay, CNRS, Laboratoire de Math\'{e}matiques d'Orsay, 91405 Orsay, France}

{"name": "Universit\u00e9 Paris-Saclay", "city": "", "country": ""}
{"name": "CNRS", "city": "", "country": ""}
{"name": "Laboratoire de Math\u00e9matiques d'Orsay", "city": "Orsay", "country": "France"}



[('2311.04844v2', 'Université Paris-Saclay', '', 'null'),
 ('2311.04844v2', 'CNRS', '', '02feahw73'),
 ('2311.04844v2',
  "Laboratoire de Mathématiques d'Orsay",
  'Orsay',
  '03ab0zs98')]

In [85]:
print(phase_one.PROMPT_TEMPLATE_V4_edited)

AttributeError: module 'phase_one_json' has no attribute 'PROMPT_TEMPLATE_V4_edited'

In [45]:
phase_one.get_single_file_results('2311.00034v1', verbose=True, vverbose=True)

Processing ftp/arxiv/papers/2311/2311.00034.tar.gz
0: \author[

```json
null
```

1: \documentclass[fleqn,usenatbib]{mnras}




\usepackage{newtxtext,newtxmath}






\usepackage[T1]{fontenc}



\DeclareRobustCommand{\VAN}[3]{#2}
\let\VANthebibliography\thebibliography
\def\thebibliography{\DeclareRobustCommand{\VAN}[3]{##3}\VANthebibliography}

\usepackage{lipsum}  



\usepackage{graphicx}	
\usepackage{amsmath}	





\newcommand{\vek}[1]{\mathbf{#1}}
\newcommand{\pdv}[2]{\frac{\partial #1}{\partial #2}}
\newcommand{\BigD}[1]{\pdv{\mathbf{#1}}{t}+(\mathbf{u}\cdot\mathbf{\nabla})\mathbf{#1}}
\newcommand{\BigDs}[1]{\pdv{#1}{t}+(\mathbf{u}\cdot\mathbf{\nabla}){#1}}
\newcommand{\rot}{\mathbf{\nabla}\times}

\newcommand{\para}{\parallel}
\newcommand{\dotM}{\dot{M}}
\newcommand{\norm}[1]{\left\lVert#1\right\rVert}
\newcommand{\tensor}[1]{\overleftrightarrow{\mathcal{#1}}}
\newcommand{\Tfrac}[2]{\left(\frac{#1}{#2}\right) }
\newcommand{\divi}{\mathbf{\nabla}\cdot}
\newcommand{\vgrad}[1]{\mat

[('2311.00034v1', 'Northwestern University', 'Evanston', '000e0be47'),
 ('2311.00034v1',
  'Institut de Recherche en Astrophysique et Planétologie',
  'Toulouse',
  '05hm2ja81'),
 ('2311.00034v1', 'Harvard University', 'Cambridge', '03vek6s52'),
 ('2311.00034v1', 'Georgia Institute of Technology', 'Atlanta', '01zkghx44')]

### Clean up results

In [ ]:
res_df = pd.read_csv("gs://institutional-extract-scratch/output/2311_db_all.csv.zip")

In [ ]:
res_df.shape
res_df.head()

In [ ]:
# probably html formatted files
error_files = res_df[res_df['name']=='error']['arx_id'].to_list()
len(error_files)

In [ ]:
no_ror_df = res_df[pd.isna(res_df['name']) | pd.isna(res_df['ror'])]
no_ror_df.shape
no_ror_df.head()

In [ ]:
nn_df = res_df[pd.isna(res_df['name'])]
nn_df.shape
nn_df.head()

In [ ]:
res_list = []
for arx in tqdm(nn_df['arx_id'][:5]):
    res_list.append(phase_one.get_single_file_results(arx))

In [ ]:
res_list

In [90]:
importlib.reload(phase_one)

AttributeError: 'rorFinder' object has no attribute 'doc_k'

### Examine extract

In [48]:
arx_id = '2311.00034v1'
phase_one.get_single_file_results(arx_id, verbose=True, vverbose=True)

Processing ftp/arxiv/papers/2311/2311.00034.tar.gz
0: \author[

```json
null
```

1: \documentclass[fleqn,usenatbib]{mnras}




\usepackage{newtxtext,newtxmath}






\usepackage[T1]{fontenc}



\DeclareRobustCommand{\VAN}[3]{#2}
\let\VANthebibliography\thebibliography
\def\thebibliography{\DeclareRobustCommand{\VAN}[3]{##3}\VANthebibliography}

\usepackage{lipsum}  



\usepackage{graphicx}	
\usepackage{amsmath}	





\newcommand{\vek}[1]{\mathbf{#1}}
\newcommand{\pdv}[2]{\frac{\partial #1}{\partial #2}}
\newcommand{\BigD}[1]{\pdv{\mathbf{#1}}{t}+(\mathbf{u}\cdot\mathbf{\nabla})\mathbf{#1}}
\newcommand{\BigDs}[1]{\pdv{#1}{t}+(\mathbf{u}\cdot\mathbf{\nabla}){#1}}
\newcommand{\rot}{\mathbf{\nabla}\times}

\newcommand{\para}{\parallel}
\newcommand{\dotM}{\dot{M}}
\newcommand{\norm}[1]{\left\lVert#1\right\rVert}
\newcommand{\tensor}[1]{\overleftrightarrow{\mathcal{#1}}}
\newcommand{\Tfrac}[2]{\left(\frac{#1}{#2}\right) }
\newcommand{\divi}{\mathbf{\nabla}\cdot}
\newcommand{\vgrad}[1]{\mat

[('2311.00034v1', 'Northwestern University', 'Evanston', '000e0be47'),
 ('2311.00034v1',
  'Institut de Recherche en Astrophysique et Planétologie (IRAP)',
  'Toulouse',
  '05hm2ja81'),
 ('2311.00034v1', 'Harvard University', 'Cambridge', '03vek6s52'),
 ('2311.00034v1', 'Georgia Institute of Technology', 'Atlanta', '01zkghx44')]

In [82]:
arx_id = '2311.00034v1'

yymm = arx_id.split(".")[0]
paper_id = arx_id.split("v")[0]
tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"
tex_main = phase_one.find_main_tex_source_in_tar(tar_path)[0]

doc = """
Parses a .tex file:
- Removes LaTeX comments
- Extracts institution names (via recursive regex)
- Extracts text before the abstract
"""
from google.cloud import storage
PROJECT_ID = "arxiv-development"
PRD_PROJECT = 'arxiv-production'
PRD_BUCKET_LOC = 'arxiv-production-data' 

client = storage.Client(project=PRD_PROJECT)
bucket = client.bucket(PRD_BUCKET_LOC)
blob = bucket.blob(tar_path)
tar_bytes = blob.download_as_bytes()
if tar_path.endswith(".tar.gz"):
    try:
        with tarfile.open(fileobj=io.BytesIO(tar_bytes), mode='r') as in_tar:
            fp = in_tar.extractfile(tex_main)
            wrapped_file = io.TextIOWrapper(fp, newline=None, encoding='utf-8') #universal newlines
            source_text = phase_one.pre_format(wrapped_file.read())
    except UnicodeDecodeError:
        try:
            with tarfile.open(fileobj=io.BytesIO(tar_bytes), mode='r') as in_tar:
                fp = in_tar.extractfile(tex_main)
                raw_data = in_tar.extractfile(tex_main).peek(10000)
                result = chardet.detect(raw_data)
                detected_encoding = result["encoding"]
                wrapped_file = io.TextIOWrapper(
                    fp, 
                    newline=None, 
                    encoding=detected_encoding, 
                    errors="replace"
                ) #universal newlines
                source_text = wrapped_file.read()
        except Exception as e:
            print(
                f"Failed to read {tar_path}-{tex_main} with"
                " detected encoding {detected_encoding}: {e}"
            )
            #return None
else:
    try:
        with gzip.open(io.BytesIO(tar_bytes), 'rt', encoding='utf-8') as in_gz:
            source_text = in_gz.read()
    except UnicodeDecodeError:
        try:
            with gzip.open(io.BytesIO(tar_bytes), 'rb') as in_gz:
                raw_data = in_gz.peek(10000)
                result = chardet.detect(raw_data)
                detected_encoding = result["encoding"]
            with gzip.open(
                io.BytesIO(tar_bytes),
                'rt', 
                encoding=detected_encoding
            ) as in_gz:
                source_text = in_gz.read()
        except Exception as e:
            print(
                f"Failed to read {tar_path}-{tex_main} with"
                " detected encoding {detected_encoding}: {e}"
            )

# Remove LaTeX comments (lines starting with non-escaped %)
content = re.sub(r"(?<!\\)%.*", "", source_text)
#res_list = []

# try parsing latex:
auth_macros = set([
    "author", "auth", "authors",
    "institute", "inst", "institution",
    "affiliation", "affil", "affiliations",
    "address",
    "cmsinstitute",
])
supstr = set([
    "\\textsuperscript",
])
latex_extracted_institutions = []
try:
    lxwkr = LatexWalker(content)
    (nodelist, pos, len_) = lxwkr.get_latex_nodes()
    focus_nodes = [
      (i,node) for i,node in enumerate(nodelist)
      if hasattr(node, "macroname") and node.macroname in auth_macros
    ]
    if focus_nodes:
        for i,node in focus_nodes:
            latex_extracted_institutions.append(node.latex_verbatim())
            try:
                idx_plus = 1
                while True:
                    if idx_plus > 10:
                        break
                    follow_node = nodelist[i+idx_plus]
                    if not isinstance(follow_node, LatexGroupNode):
                        idx_plus += 1
                    if isinstance(follow_node, LatexGroupNode):
                        latex_extracted_institutions.append(follow_node.latex_verbatim())
                        break
            except IndexError:
                pass
        if any(pat in lx for lx in latex_extracted_institutions for pat in supstr):
            sup_res = extract_texsuperscript(nodelist)
            latex_extracted_institutions.extend(sup_res)
    else:
        doc = [
            node for node in nodelist
            if isinstance(node, LatexEnvironmentNode) and node.environmentname=='document'
        ]
        if doc:
            focus_doc_nodes = [
              (i,node) for i, node in enumerate(doc[0].nodelist)
              if isinstance(node, LatexMacroNode) and node.macroname in auth_macros
            ]
            for i, node in focus_doc_nodes:
                latex_extracted_institutions.append(node.latex_verbatim())
                try:
                    idx_plus = 1
                    while True:
                        if idx_plus > 10:
                            break
                        follow_node = nodelist[i+idx_plus]
                        if not isinstance(follow_node, LatexGroupNode):
                            idx_plus += 1
                        if isinstance(follow_node, LatexGroupNode):
                            latex_extracted_institutions.append(follow_node.latex_verbatim())
                except IndexError:
                    pass
            if any(pat in lx for lx in latex_extracted_institutions for pat in supstr):
                sup_res = extract_texsuperscript(doc[0].nodelist)
                latex_extracted_institutions.extend(sup_res)
    if latex_extracted_institutions:
        #res_list.append(latex_extracted_institutions)
        #yield "\n".join(latex_extracted_institutions)
        pass
except Exception as e:
    print(f"Overly broad except in extract_pre_abstract_content(): {e}")
    pass

#  "recursive" regex:
#   ((?>[^{}]+|\{(?1)\})*)
# optional brackets
#   (:?\[\d+\])?\s*
# This matches text possibly containing normal characters or nested braces,
# until the outermost braces are matched.
# If your LaTeX does not have deep nesting, this mainly ensures things like $^{1}$ are correctly parsed.
institution_patterns = [
    r"\\affiliation\s*(:?\[\d+\])?\s*\{((?>[^{}]+|\{(?1)\})*)\}",
    r"\\institute\s*(:?\[\d+\])?\s*\{((?>[^{}]+|\{(?1)\})*)\}",
    r"\\address\s*(:?\[\d+\])?\s*\{((?>[^{}]+|\{(?1)\})*)\}",
    r"\\inst\s*(:?\[\d+\])?\s*\{((?>[^{}]+|\{(?1)\})*)\}",
    r"\\affil\s*(:?\[\d+\])?\s*\{((?>[^{}]+|\{(?1)\})*)\}",
    r"\\author\s*(:?\[\d+\])?\s*{[^}]+}{([^}]+)}",
    r"\\cmsinstitute\s*(:?\[\d+\])?\s*{[^}]+}{([^}]+)}",
]

extracted_institutions = []
for pattern in institution_patterns:
    # Use regex.findall with DOTALL to allow '.' to match newlines
    matches = re.findall(pattern, content, flags=re.DOTALL)
    if matches:
        # Strip each match and add to list
        for m in matches:
            if isinstance(m, tuple):
                extracted_institutions.append(" ".join(m_i for m_i in m))
            else:
                extracted_institutions.extend(m.strip() for m in matches if m.strip())

# If any institution info is extracted, return the deduplicated joined text
if extracted_institutions:
    # You can change the join method; here we join by newline and use set to deduplicate
    #return "\n".join(set(extracted_institutions))
    #res_list.append("\n".join(set(extracted_institutions)))
    #yield "\n".join(set(extracted_institutions))
    pass

# If no institution found, try extracting the text before the abstract
match = re.split(
    r"\\begin\s*{\s*abstract\s*}|\\s*\\section\s*{\s*Abstract\s*}",
    content,
    maxsplit=1,
    flags=re.IGNORECASE
)
if len(match) > 1:
    #return match[0].strip()
    #res_list.append(match[0].strip())
    #yield match[0].strip()
    pass

# If still not found, return the first 1/3 of the content as a fallback
content_length = len(content)
if content_length > 0:
    one_third_length = max(content_length//3, 2000)
    #return content[:one_third_length].strip()
    #res_list.append(content[:one_third_length].strip())
    #yield content[:one_third_length].strip()
    pass

# If still not found, return an empty string
#if res_list:
#  yield res_list
#else:
# yield ["",]



In [50]:
focus_nodes

[(104,
  LatexMacroNode(parsing_state=<parsing state 139947566290528>, pos=2185, len=8, macroname='author', nodeargd=ParsedMacroArgs(argspec='{', argnlist=[LatexCharsNode(parsing_state=<parsing state 139947566290528>, pos=2192, len=1, chars='[')]), macro_post_space=''))]

In [80]:
nodelist[106]

LatexGroupNode(parsing_state=<parsing state 139947566290528>, pos=2217, len=701, nodelist=[LatexCharsNode(parsing_state=<parsing state 139947566290528>, pos=2218, len=23, chars='\nJonatan Jacquemin-Ide,'), LatexMathNode(parsing_state=<parsing state 139947566290528>, pos=2241, len=6, displaytype='inline', nodelist=[LatexCharsNode(parsing_state=<parsing state 139947566439760>, pos=2242, len=1, chars='^'), LatexGroupNode(parsing_state=<parsing state 139947566439760>, pos=2243, len=3, nodelist=[LatexCharsNode(parsing_state=<parsing state 139947566439760>, pos=2244, len=1, chars='1')], delimiters=('{', '}'))], delimiters=('$', '$')), LatexMacroNode(parsing_state=<parsing state 139947566290528>, pos=2247, len=7, macroname='thanks', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''), LatexGroupNode(parsing_state=<parsing state 139947566290528>, pos=2254, len=47, nodelist=[LatexCharsNode(parsing_state=<parsing state 139947566290528>, pos=2255, len=45, chars='E-mail: jonata

In [76]:
[x.parsing_state for x in nodelist[104:115]]

In [83]:
latex_extracted_institutions

['\\author[',
 '{\nJonatan Jacquemin-Ide,$^{1}$\\thanks{E-mail: jonatan.jacqueminide@northwestern.edu}\nFrançois Rincon, $^{2}$\nAlexander Tchekhovskoy,$^{1}$\nand Matthew Liska$^{3,4}$\n\\\\\n\n$^{1}$Center for Interdisciplinary Exploration $\\&$ Research in Astrophysics (CIERA), Physics and Astronomy, Northwestern University, Evanston, IL 60202, USA\\\\\n$^{2}$Institut de Recherche en Astrophysique et Planétologie (IRAP), Université de Toulouse, CNRS, UPS, Toulouse, France\\\\\n$^{3}$Institute for Theory and Computation, Harvard University, 60 Garden Street, Cambridge, MA 02138, USA\\\\\n$^{4}$Center for Relativistic Astrophysics, Georgia Institute of Technology, Howey Physics Bldg, 837 State St NW, Atlanta, GA 30332, USA\\\\\n}']

In [81]:
macro_node=nodelist[104]
arg_list = macro_node.nodeargd.argnlist
arg_node = arg_list[0]
isinstance(arg_node, LatexCharsNode)

True

In [ ]:
def get_arg_contents(node_list, macro_idx):
    macro_node = nodelist[macro_idx]
    arg_list = None
    
    try:
        arg_list = macro_node.nodeargd.argnlist
        if not arg_list or len(arg_list) > 1:
            return ""
        arg_node = arg_list[0]
        if 
    except AttributeError:
        pass
    
    
        

    
    

In [177]:
def extract_texsuperscript(latex_node_list, res=None):
    '''for each superscript, get the contents of the next LatexCharsNode'''
    bailout_macros = set(['abstract', 'subsection'])
    if res is None:
        res = []
    for i, node in enumerate(latex_node_list):
        sublist = []
        #print(type(node))
        try:
            if node.macroname=='textsuperscript':
                run_started = False
                text_list = []
                for nnode in latex_node_list[i:]:
                    #print(type(nnode))
                    if isinstance(nnode, LatexCharsNode):
                        text_list.append(nnode.latex_verbatim())
                        run_started = True
                    elif isinstance(nnode, LatexMacroNode):
                        if nnode.macroname == '&':
                            text_list.append('&')
                        elif run_started:
                            break
                    elif not isinstance(nnode, LatexCharsNode):
                        if run_started:
                            break
                if text_list:
                    res.append(" ".join(text_list))
        except AttributeError:
            pass
        if isinstance(node, LatexMacroNode):
            try: 
                if node.macroname in bailout_macros:
                    break
                sublist = node.nodeargd.argnlist
                #print(sublist)
            except AttributeError:
                pass
        if isinstance(node, (LatexGroupNode, LatexEnvironmentNode)):
            try:
                sublist = node.nodelist
                #print(sublist)
            except AttributeError:
                pass
        if isinstance(node, LatexEnvironmentNode) and node.environmentname=='document':
            break
        if sublist:
            extract_texsuperscript(sublist, res)
    return res

In [162]:
for node in nodelist[139].nodeargd.argnlist[0].nodelist:
    print(node)
    print('\n')
    print('--')
    print('\n')

LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=2703, len=16, macroname='textsuperscript', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space='')


--


LatexGroupNode(parsing_state=<parsing state 140674525854880>, pos=2719, len=6, nodelist=[LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=2720, len=4, macroname='dag', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space='')], delimiters=('{', '}'))


--


LatexCharsNode(parsing_state=<parsing state 140674525854880>, pos=2725, len=21, chars='Corresponding authors')


--




In [163]:
extract_texsuperscript(nodelist[139:140])  #nodelist[138:150])

['Corresponding authors']

[LatexGroupNode(parsing_state=<parsing state 140674525854880>, pos=2702, len=45, nodelist=[LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=2703, len=16, macroname='textsuperscript', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''), LatexGroupNode(parsing_state=<parsing state 140674525854880>, pos=2719, len=6, nodelist=[LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=2720, len=4, macroname='dag', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space='')], delimiters=('{', '}')), LatexCharsNode(parsing_state=<parsing state 140674525854880>, pos=2725, len=21, chars='Corresponding authors')], delimiters=('{', '}'))]

In [167]:
extract_texsuperscript(nodelist[139:])

['Corresponding authors',
 'Molecular Robotics, Kerala, India; ',
 'School of Computing and Information, University of Pittsburgh, Pennsylvania, PA, USA; ',
 'Health Informatics ',
 'Department of Computer Science, Courant Institute of Mathematical Sciences, New York University, New York, NY, USA; ',
 'Institute for Next Generation Healthcare, Mount Sinai Health System, New York, NY, USA; ',
 'Department of Preventive Cardiology, DeBakey Heart ',
 'Hasso Plattner Institute for Digital Health, Icahn School of Medicine at Mount Sinai, New York, NY, USA; ',
 'Faculty of Medicine, Imperial College London, London, UK; ',
 'School of Engineering Medicine,  Texas A',
 'Department of Translational Medical Sciences, Center for Genomic and Precision Medicine, Texas A']

In [103]:
extra_pats = ["\\textsuperscript"]

[pat in l for l in latex_extracted_institutions for pat in extra_pats]

[True]

In [115]:
focus_nodes

[(137,
  LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=1902, len=794, macroname='author', nodeargd=ParsedMacroArgs(argspec='{', argnlist=[LatexGroupNode(parsing_state=<parsing state 140674525854880>, pos=1909, len=787, nodelist=[LatexCharsNode(parsing_state=<parsing state 140674525854880>, pos=1910, len=28, chars='\n    Sonish Sivarajkumar, MS'), LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=1938, len=16, macroname='textsuperscript', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''), LatexGroupNode(parsing_state=<parsing state 140674525854880>, pos=1954, len=3, nodelist=[LatexCharsNode(parsing_state=<parsing state 140674525854880>, pos=1955, len=1, chars='1')], delimiters=('{', '}')), LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=1957, len=16, macroname='textsuperscript', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''), LatexGroupNode(parsing_state=<parsing state 140674525854880>, 

In [111]:
nodelist[141].nodeargd.argnlist[0].nodelist

[LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=2756, len=16, macroname='textsuperscript', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''),
 LatexGroupNode(parsing_state=<parsing state 140674525854880>, pos=2772, len=3, nodelist=[LatexCharsNode(parsing_state=<parsing state 140674525854880>, pos=2773, len=1, chars='1')], delimiters=('{', '}')),
 LatexCharsNode(parsing_state=<parsing state 140674525854880>, pos=2775, len=35, chars='Molecular Robotics, Kerala, India; '),
 LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=2810, len=2, macroname='\n', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''),
 LatexCharsNode(parsing_state=<parsing state 140674525854880>, pos=2812, len=6, chars='      '),
 LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=2818, len=16, macroname='textsuperscript', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''),
 LatexGroupNode(parsing_state=<pars

## Test

In [ ]:
os.cpu_count()

In [ ]:
# importlib.reload(phase_one)

## times

```
Batch size    parallel workers    thread workers    time               n       sec/item        errors
   5             2                   5                                 100     
  10             2                   5                                 100     
  10             2                  10                                 100     
  
   5             3                   5                2 min 50s        100     1.70
  10             3                   5                2 min  8s        100     1.28
  10             3                  10                2 min 23s        100     1.43
  
  10             8                  10                3min 44s        1000      .22              0
  20             8                  10                3min 15s.       1000      .21              0       6414, 255; 6717, 296
  20             8                  20                3min 27s        1000      .21              0
  50             8                  25                3min 21s        1000      .20 
  
  25            12                  25                8min 1s         1000      .48
 100            12                  25                12min 39s       1000      .73 
 

10                   3                          5                    1 min 14s
```

In [ ]:
%%time

test_ids_df = pd.read_csv("gs://institutional-extract-scratch/reference/arx_ids/2311_ids.csv")
#test_ids_df.head()
ids_2311_all = test_ids_df["arx_id"].unique()


tt = TicToc()

input_ids = set(ids_2311_all)
save_name = "2311_db"
sample_size = "all"
batch_size = 20
parallel_workers = 8
thread_workers = 10

def worker(arx_id_list):
    res = phase_one.process_tex_files(arx_id_list, max_workers=thread_workers)
    return res

def run_phase_one_in_parallel(arx_id_batches):
    res_list = []
    total_len = sum(len(x) for x in arx_id_batches)
    with concurrent.futures.ProcessPoolExecutor(max_workers=parallel_workers) as executor:
        futures = [executor.submit(worker, arx_id_list) for arx_id_list in arx_id_batches]
        for future in as_completed(futures): #tqdm(as_completed(futures), total=len(futures)):
            res = future.result()
            res_list.extend(res)
#        res = [future.result() for future in concurrent.futures.as_completed(futures)]
#        for batch in res:
#            res_list.extend(batch)
    return res_list

def format_results(arxid_inst_ror_list):
    res_list = []
    for key, group in tqdm(itr.groupby(arxid_inst_ror_list, key=lambda x: x[0])):
        ror_inst = []
        for x in group:
            clean_name = x[1].split('.', 1)[-1].strip()
            ror = 'null'
            if len(x) == 3:
                ror = x[2].strip()
            inst = {
                'name':clean_name,
                'ror_id':ror
            }
            ror_inst.append(inst)
        arx_rec = {
            "arxiv_id": key.strip(),
            "institutions_with_ror": ror_inst,
        }
        res_list.append(arx_rec)
    return res_list



try:
    objects = []
    with open(f"checkpoints/{save_name}_{sample_size}.pkl", 'rb') as cp_fp:
        while True:
            try:
                obj = pickle.load(cp_fp)
                objects.append(obj)
            except EOFError:
                break
except FileNotFoundError as e:
    pass

known_ids = []
known_res = []
for obj in objects:
    known_ids.extend(x[0] for x in obj)
    known_res.extend(obj)

known_ids = set(known_ids)
input_ids = input_ids - known_ids
print(f"Found checkpoints for {len(known_ids)} nodes.")

if sample_size != "all":
    input_ids = input_ids[:sample_size]

#import concurrent.futures
#import phase_one


os.environ["TOKENIZERS_PARALLELISM"] = "false" 

batches = np.array_split(list(input_ids), len(input_ids)//batch_size)
tt.tic()
print(f"Start: {len(input_ids)} in {len(batches)} batches")
with open(f"checkpoints/{save_name}_{sample_size}.pkl", 'ab') as cp_fp:
    res = run_phase_one_in_parallel(batches, cp_fp)
tt.toc()
known_res.extend(res)
res_df = pd.DataFrame.from_records(known_res, columns=['arx_id', 'name', 'ror'])
res_df.to_csv(f"gs://institutional-extract-scratch/output/{save_name}_{sample_size}.csv.zip", index=False)
#res_list = format_results(res)
tt.toc()

In [ ]:
len(res)
sum( 1 for x in res if x[1] == 'error' )
sum( 1 for x in res if x[2] == 'null' )

In [ ]:
len(res)
sum( 1 for x in res if x[1] == 'error' )
sum( 1 for x in res if x[2] == 'null' )

In [ ]:
res_df = pd.DataFrame.from_records(res, columns=['arx_id', 'name', 'ror'])
res_df.to_csv(f"gs://institutional-extract-scratch/output/2311_scopus_{sample_size}.csv.zip", index=False)

In [ ]:
res

In [ ]:
%%time

tt = TicToc()

sample_size = 100
batch_size = 20
parallel_workers = 3
thread_workers = 20
#import concurrent.futures
#import phase_one

def worker(arx_id_list):
    res = phase_one.process_tex_files(arx_id_list, max_workers=thread_workers)
    return res

def run_phase_one_in_parallel(arx_id_batches):
    res_list = []
    total_len = sum(len(x) for x in arx_id_batches)
    with concurrent.futures.ProcessPoolExecutor(max_workers=parallel_workers) as executor:
        futures = [executor.submit(worker, arx_id_list) for arx_id_list in arx_id_batches]
        res = [future.result() for future in concurrent.futures.as_completed(futures)]
        for batch in res:
            res_list.extend(batch)
    return res_list

def run_phase_two_in_sequence(arxid_inst_list):
    res_list = []
    for key, group in tqdm(itr.groupby(arxid_inst_list, key=lambda x: x[0])):
        ror_inst = []
        for x in group:
            clean_name = x[1].split('.', 1)[-1].strip()
            ror = get_ror(clean_name)
            inst = {
                'name':clean_name,
                'ror_id':ror
            }
            ror_inst.append(inst)
        arx_rec = {
            "arxiv_id": key.strip(),
            "institutions_with_ror": ror_inst,
        }
        res_list.append(arx_rec)
    return res_list

os.environ["TOKENIZERS_PARALLELISM"] = "false" 
input_ids = scopus_all[:sample_size]
batches = np.array_split(input_ids, len(input_ids)//batch_size)
tt.tic()
print(f"Start Phase 1, {len(input_ids)} in {len(batches)} batches")
res = run_phase_one_in_parallel(batches)
tt.toc()
res[:5]
print("Start Phase 2")
res_list = run_phase_two_in_sequence(res)
tt.toc()
res_list[:5]

In [ ]:
%%time

tt = TicToc()

sample_size = 100
batch_size = 20
parallel_workers = 3
thread_workers = 20
#import concurrent.futures
#import phase_one

def worker(arx_id_list):
    res = phase_one.process_tex_files(arx_id_list, max_workers=thread_workers)
    print(len(res))
    res_list = run_phase_two_in_sequence(res)
    return res_list

def run_phase_one_in_parallel(arx_id_batches):
    res_list = []
    total_len = sum(len(x) for x in arx_id_batches)
    with concurrent.futures.ProcessPoolExecutor(max_workers=parallel_workers) as executor:
        futures = [executor.submit(worker, arx_id_list) for arx_id_list in arx_id_batches]
        res = [future.result() for future in concurrent.futures.as_completed(futures)]
        for batch in res:
            res_list.extend(batch)
    return res_list

def run_phase_two_in_sequence(arxid_inst_list):
    res_list = []
    for key, group in itr.groupby(arxid_inst_list, key=lambda x: x[0]):
        ror_inst = []
        for x in group:
            clean_name = x[1].split('.', 1)[-1].strip()
            ror = get_ror(clean_name)
            inst = {
                'name':clean_name,
                'ror_id':ror
            }
            ror_inst.append(inst)
        arx_rec = {
            "arxiv_id": key.strip(),
            "institutions_with_ror": ror_inst,
        }
        res_list.append(arx_rec)
    return res_list


os.environ["TOKENIZERS_PARALLELISM"] = "false" 
input_ids = scopus_all[:sample_size]
batches = np.array_split(input_ids, len(input_ids)//batch_size)
tt.tic()
print(f"Start Phase 1, {len(input_ids)} in {len(batches)} batches")
res = run_phase_one_in_parallel(batches)
tt.toc()
res[:5]
#print("Start Phase 2")
#res_list = run_phase_two_in_sequence(res)
#res_list[:5]

In [ ]:
res[:5]

In [ ]:
res_list

## ROR Index experiments

In [107]:
ror_gspath = 'gs://institutional-extract-scratch/reference/v1.63-2025-04-03-ror-data_schema_v2.json'
fs = gcsfs.GCSFileSystem()
with fs.open(ror_gspath, "r", encoding="utf-8") as f:
    ror_data = json.load(f)

In [111]:
for i,entry in tqdm(enumerate(ror_data)):
    ror_id = entry.get("id", "")
    if ror_id.endswith('00jjx8s55'):
        break

0it [00:00, ?it/s]

In [112]:
for i,entry in tqdm(enumerate(ror_data)):
    ror_id = entry.get("id", "")
    if ror_id.endswith('00jjx8s55'):
        break

{'admin': {'created': {'date': '2018-11-14', 'schema_version': '1.0'},
  'last_modified': {'date': '2025-03-26', 'schema_version': '2.1'}},
 'domains': [],
 'established': 1945,
 'external_ids': [{'all': ['501100006489'],
   'preferred': None,
   'type': 'fundref'},
  {'all': ['grid.5583.b'], 'preferred': 'grid.5583.b', 'type': 'grid'},
  {'all': ['0000 0001 2299 8025'], 'preferred': None, 'type': 'isni'},
  {'all': ['Q868550'], 'preferred': None, 'type': 'wikidata'}],
 'id': 'https://ror.org/00jjx8s55',
 'links': [{'type': 'website', 'value': 'http://www.cea.fr/'},
  {'type': 'wikipedia',
   'value': 'https://en.wikipedia.org/wiki/French_Alternative_Energies_and_Atomic_Energy_Commission'}],
 'locations': [{'geonames_details': {'continent_code': 'EU',
    'continent_name': 'Europe',
    'country_code': 'FR',
    'country_name': 'France',
    'country_subdivision_code': 'IDF',
    'country_subdivision_name': 'Île-de-France',
    'lat': 48.85341,
    'lng': 2.3488,
    'name': 'Paris'},


## Scratch